# NOTEBOOK 07 — MODEL DESIGN AND EXPERIMENTAL SETUP

## 1. Mục tiêu

Khảo sát, thiết kế và lựa chọn kiến trúc học sâu phù hợp với bài toán phát hiện, định vị tổn thương Mass trên ảnh X-quang tuyến vú.

Xác định phương pháp xử lý ảnh đầu vào, cấu trúc đầu ra, chiến lược kết hợp Object Detection và Pathology Classification, đồng thời thiết lập cấu hình thực nghiệm phục vụ quá trình huấn luyện tại Notebook 08.

## 2. Phạm vi dữ liệu

Sử dụng Detection Dataset được xây dựng tại Notebook 06 từ bộ dữ liệu CBIS-DDSM.

- Tổng số Annotation: 1.696.
- Ảnh Full Image duy nhất: 1.592.
- Patient ID: 892.
- Train: 1.054 Annotation.
- Validation: 264 Annotation.
- Test: 378 Annotation.
- Loại tổn thương: Mass.
- Nhãn Pathology: BENIGN, BENIGN_WITHOUT_CALLBACK và MALIGNANT.

## 3. Nguyên tắc thực hiện

- Lựa chọn kiến trúc dựa trên đặc điểm dữ liệu, mục tiêu nghiên cứu và tài nguyên phần cứng.
- Ưu tiên bảo toàn thông tin hình thái và vị trí tổn thương trên ảnh mammography.
- Đánh giá ảnh hưởng của resize, cropping và tiling đối với khả năng phát hiện tổn thương.
- Phân biệt rõ nhiệm vụ Object Detection và Pathology Classification.
- Bảo toàn thông tin Pathology và có căn cứ khi xử lý nhãn BENIGN_WITHOUT_CALLBACK.
- Thiết kế thực nghiệm có thể tái lập, kiểm soát dữ liệu và tránh rò rỉ giữa các tập.

## 4. Giới hạn

- Tập thực nghiệm hiện tại chỉ tập trung vào tổn thương Mass.
- Số lượng Annotation và ảnh có hạn, cần cân nhắc nguy cơ overfitting.
- Ảnh mammography có độ phân giải cao và tổn thương có kích thước khác nhau, gây thách thức về khả năng phát hiện và bộ nhớ GPU.
- Bounding Box được suy ra từ ROI Mask của CBIS-DDSM.
- Kiến trúc và cấu hình cuối cùng chỉ được xác định sau khi hoàn thành khảo sát và đánh giá tính khả thi.

## 5. Phạm vi xử lý

Notebook thực hiện kiểm tra Detection Dataset, khảo sát kiến trúc Object Detection, so sánh phương pháp xử lý ảnh, thiết kế đầu ra phát hiện và phân loại Pathology, lựa chọn chiến lược thực nghiệm và xác định cấu hình mô hình.

Không thực hiện huấn luyện mô hình chính thức trong Notebook này. Quá trình huấn luyện và tối ưu hóa được thực hiện tại Notebook 08.

In [ ]:
# 07.1 — KIỂM TRA DETECTION DATASET VÀ XÁC ĐỊNH BÀI TOÁN

import pandas as pd
import numpy as np
from pathlib import Path

# 1. Cấu hình đường dẫn

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

NB067_DATASET_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb06_detection_dataset"
    / "nb06_7_detection_dataset"
)

NB07_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb07_model_design"
)

IMAGE_MANIFEST_FILE = (
    NB067_DATASET_DIR / "nb06_7_detection_image_manifest.csv"
)

ANNOTATION_FILE = (
    NB067_DATASET_DIR / "nb06_7_detection_annotations.csv"
)

COMPLETE_DATASET_FILE = (
    NB067_DATASET_DIR / "nb06_7_detection_dataset_complete.csv"
)

NB07_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Kiểm tra file đầu vào

input_files = {
    "image_manifest": IMAGE_MANIFEST_FILE,
    "detection_annotations": ANNOTATION_FILE,
    "complete_dataset": COMPLETE_DATASET_FILE
}

input_file_audit_df = pd.DataFrame([
    {
        "data_file": name,
        "file_path": str(path),
        "exists": path.is_file(),
        "file_size_mb": (
            round(path.stat().st_size / (1024 ** 2), 3)
            if path.is_file() else np.nan
        )
    }
    for name, path in input_files.items()
])

missing_required_files = [
    name
    for name in ["image_manifest", "detection_annotations"]
    if not input_files[name].is_file()
]

if missing_required_files:
    raise FileNotFoundError(
        f"Thiếu file dữ liệu bắt buộc: {missing_required_files}"
    )

# 3. Đọc dữ liệu

detection_images_df = pd.read_csv(IMAGE_MANIFEST_FILE)
detection_labels_df = pd.read_csv(ANNOTATION_FILE)

# 4. Kiểm tra cấu trúc dữ liệu

print("IMAGE MANIFEST")
print("Shape:", detection_images_df.shape)
print("Columns:", detection_images_df.columns.tolist())

print("\nDETECTION ANNOTATIONS")
print("Shape:", detection_labels_df.shape)
print("Columns:", detection_labels_df.columns.tolist())

# 5. Kiểm tra các trường dữ liệu bắt buộc

image_required_columns = [
    "image_id",
    "normalized_image_path",
    "patient_id",
    "dataset_split",
    "image_view",
    "image_height",
    "image_width",
    "annotation_count"
]

annotation_required_columns = [
    "annotation_id",
    "image_id",
    "patient_id",
    "dataset_split",
    "pathology",
    "x_min",
    "y_min",
    "x_max",
    "y_max"
]

missing_image_columns = [
    col for col in image_required_columns
    if col not in detection_images_df.columns
]

missing_annotation_columns = [
    col for col in annotation_required_columns
    if col not in detection_labels_df.columns
]

if missing_image_columns:
    raise ValueError(
        f"Image Manifest thiếu cột: {missing_image_columns}"
    )

if missing_annotation_columns:
    raise ValueError(
        f"Detection Annotations thiếu cột: {missing_annotation_columns}"
    )

# 6. Tổng quan dữ liệu

dataset_overview_df = pd.DataFrame([
    {
        "component": "Image Manifest",
        "total_records": len(detection_images_df),
        "unique_ids": detection_images_df["image_id"].nunique()
    },
    {
        "component": "Detection Annotations",
        "total_records": len(detection_labels_df),
        "unique_ids": detection_labels_df["annotation_id"].nunique()
    },
    {
        "component": "Patient",
        "total_records": detection_labels_df["patient_id"].nunique(),
        "unique_ids": detection_labels_df["patient_id"].nunique()
    }
])

# 7. Phân bố Dataset Split

split_summary_df = (
    detection_labels_df.groupby("dataset_split")
    .agg(
        annotation_count=("annotation_id", "nunique"),
        image_count=("image_id", "nunique"),
        patient_count=("patient_id", "nunique")
    )
    .reset_index()
)

# 8. Phân bố Pathology

pathology_summary_df = (
    detection_labels_df.groupby(
        ["dataset_split", "pathology"],
        dropna=False
    )
    .agg(
        annotation_count=("annotation_id", "nunique"),
        image_count=("image_id", "nunique"),
        patient_count=("patient_id", "nunique")
    )
    .reset_index()
)

# 9. Phân bố Image View

view_summary_df = (
    detection_images_df.groupby(
        ["dataset_split", "image_view"],
        dropna=False
    )
    .agg(
        image_count=("image_id", "nunique"),
        patient_count=("patient_id", "nunique")
    )
    .reset_index()
)

# 10. Thống kê kích thước ảnh

image_dimension_summary_df = (
    detection_images_df.groupby(
        ["image_height", "image_width"],
        dropna=False
    )
    .agg(
        image_count=("image_id", "nunique")
    )
    .reset_index()
    .sort_values("image_count", ascending=False)
)

image_dimension_statistics_df = (
    detection_images_df[
        ["image_height", "image_width"]
    ]
    .describe()
    .T
    .reset_index(names="dimension")
)

# 11. Thống kê Bounding Box

bbox_columns = [
    "x_min",
    "y_min",
    "x_max",
    "y_max"
]

bbox_numeric_df = detection_labels_df[
    bbox_columns
].apply(pd.to_numeric, errors="coerce")

bbox_size_df = detection_labels_df[
    [
        "annotation_id",
        "image_id",
        "patient_id",
        "dataset_split",
        "pathology"
    ]
].copy()

bbox_size_df["bbox_width"] = (
    bbox_numeric_df["x_max"] - bbox_numeric_df["x_min"]
)

bbox_size_df["bbox_height"] = (
    bbox_numeric_df["y_max"] - bbox_numeric_df["y_min"]
)

bbox_size_df["bbox_area"] = (
    bbox_size_df["bbox_width"] * bbox_size_df["bbox_height"]
)

bbox_size_df["bbox_valid"] = (
    bbox_size_df["bbox_width"].gt(0)
    & bbox_size_df["bbox_height"].gt(0)
    & bbox_numeric_df.notna().all(axis=1)
)

bbox_statistics_df = (
    bbox_size_df[
        ["bbox_width", "bbox_height", "bbox_area"]
    ]
    .describe()
    .T
    .reset_index(names="metric")
)

bbox_quality_summary_df = pd.DataFrame([
    {
        "metric": "total_annotations",
        "value": len(bbox_size_df)
    },
    {
        "metric": "valid_bounding_boxes",
        "value": int(bbox_size_df["bbox_valid"].sum())
    },
    {
        "metric": "invalid_bounding_boxes",
        "value": int((~bbox_size_df["bbox_valid"]).sum())
    },
    {
        "metric": "missing_bbox_coordinates",
        "value": int(bbox_numeric_df.isna().any(axis=1).sum())
    }
])

# 12. Kiểm tra quan hệ Image và Annotation

image_ids = set(
    detection_images_df["image_id"].dropna()
)

annotation_image_ids = set(
    detection_labels_df["image_id"].dropna()
)

missing_image_mapping_count = len(
    annotation_image_ids - image_ids
)

unannotated_image_count = len(
    image_ids - annotation_image_ids
)

duplicate_image_ids = int(
    detection_images_df["image_id"].duplicated().sum()
)

duplicate_annotation_ids = int(
    detection_labels_df["annotation_id"].duplicated().sum()
)

# 13. Kiểm tra đường dẫn ảnh đã chuẩn hóa

image_path_exists_count = int(
    detection_images_df["normalized_image_path"]
    .apply(
        lambda path: (
            Path(str(path)).is_file()
            if pd.notna(path) else False
        )
    )
    .sum()
)

# 14. Kiểm tra tính nhất quán Patient và Dataset Split

patient_split_counts_df = (
    detection_labels_df.groupby("patient_id")["dataset_split"]
    .nunique()
)

patient_split_leakage_count = int(
    (patient_split_counts_df > 1).sum()
)

# 15. Kiểm tra nhãn Pathology

expected_pathology_labels = {
    "BENIGN",
    "BENIGN_WITHOUT_CALLBACK",
    "MALIGNANT"
}

observed_pathology_labels = set(
    detection_labels_df["pathology"].dropna().unique()
)

unexpected_pathology_labels = (
    observed_pathology_labels - expected_pathology_labels
)

missing_pathology_labels = (
    expected_pathology_labels - observed_pathology_labels
)

missing_pathology_count = int(
    detection_labels_df["pathology"].isna().sum()
)

# 16. Tổng hợp báo cáo toàn vẹn dữ liệu

input_integrity_records = [
    ("unique_images", detection_images_df["image_id"].nunique(), 1592),
    ("unique_annotations", detection_labels_df["annotation_id"].nunique(), 1696),
    ("unique_patients", detection_labels_df["patient_id"].nunique(), 892),
    ("duplicate_image_ids", duplicate_image_ids, 0),
    ("duplicate_annotation_ids", duplicate_annotation_ids, 0),
    ("missing_image_mapping", missing_image_mapping_count, 0),
    ("unannotated_images", unannotated_image_count, 0),
    ("image_path_exists", image_path_exists_count, len(detection_images_df)),
    ("patient_split_leakage", patient_split_leakage_count, 0),
    ("valid_bounding_boxes", int(bbox_size_df["bbox_valid"].sum()), len(detection_labels_df)),
    ("missing_pathology_labels", missing_pathology_count, 0),
    ("unexpected_pathology_labels", len(unexpected_pathology_labels), 0)
]

input_integrity_df = pd.DataFrame(
    input_integrity_records,
    columns=["metric", "value", "expected"]
)

input_integrity_df["status"] = np.where(
    input_integrity_df["value"] == input_integrity_df["expected"],
    "PASS",
    "CHECK"
)

pathology_label_audit_df = pd.DataFrame({
    "label": sorted(expected_pathology_labels),
    "present_in_dataset": [
        label in observed_pathology_labels
        for label in sorted(expected_pathology_labels)
    ]
})

# 17. Xuất báo cáo

output_files = {
    "nb07_1_input_file_audit.csv": input_file_audit_df,
    "nb07_1_dataset_overview.csv": dataset_overview_df,
    "nb07_1_split_summary.csv": split_summary_df,
    "nb07_1_pathology_summary.csv": pathology_summary_df,
    "nb07_1_view_summary.csv": view_summary_df,
    "nb07_1_image_dimension_summary.csv": image_dimension_summary_df,
    "nb07_1_image_dimension_statistics.csv": image_dimension_statistics_df,
    "nb07_1_bbox_statistics.csv": bbox_statistics_df,
    "nb07_1_bbox_quality_summary.csv": bbox_quality_summary_df,
    "nb07_1_pathology_label_audit.csv": pathology_label_audit_df,
    "nb07_1_input_integrity.csv": input_integrity_df
}

for filename, dataframe in output_files.items():
    dataframe.to_csv(
        NB07_OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 18. Hiển thị kết quả

print("\nKIỂM TRA FILE ĐẦU VÀO")
display(input_file_audit_df)

print("\nTỔNG QUAN DETECTION DATASET")
display(dataset_overview_df)

print("\nPHÂN BỐ DATASET SPLIT")
display(split_summary_df)

print("\nPHÂN BỐ PATHOLOGY")
display(pathology_summary_df)

print("\nPHÂN BỐ IMAGE VIEW")
display(view_summary_df)

print("\nPHÂN BỐ KÍCH THƯỚC ẢNH")
display(image_dimension_summary_df)

print("\nTHỐNG KÊ KÍCH THƯỚC ẢNH")
display(image_dimension_statistics_df)

print("\nTHỐNG KÊ BOUNDING BOX")
display(bbox_statistics_df)

print("\nKIỂM TRA CHẤT LƯỢNG BOUNDING BOX")
display(bbox_quality_summary_df)

print("\nKIỂM TRA PATHOLOGY LABEL")
display(pathology_label_audit_df)

print("\nKIỂM TRA TÍNH TOÀN VẸN ĐẦU VÀO")
display(input_integrity_df)

print(
    "\nTỔNG PASS:",
    int((input_integrity_df["status"] == "PASS").sum())
)

print(
    "TỔNG CHECK:",
    int((input_integrity_df["status"] == "CHECK").sum())
)

print("\nĐÃ LƯU KẾT QUẢ TẠI:")
print(NB07_OUTPUT_DIR)


IMAGE MANIFEST
Shape: (1592, 12)
Columns: ['image_id', 'full_image_path', 'normalized_image_path', 'patient_id', 'dataset_split', 'image_view', 'laterality', 'image_height', 'image_width', 'annotation_count', 'normalized_image_copy_count', 'shared_image']

DETECTION ANNOTATIONS
Shape: (1696, 30)
Columns: ['annotation_id', 'image_id', 'patient_id', 'dataset_split', 'normalized_image_path', 'image_height', 'image_width', 'image_view', 'laterality', 'lesion_label', 'lesion_label_id', 'pathology', 'pathology_label_id', 'lesion_candidate_id', 'abnormality_id', 'mass_shape', 'mass_margins', 'x_min', 'y_min', 'x_max', 'y_max', 'bbox_width', 'bbox_height', 'bbox_center_x', 'bbox_center_y', 'bbox_x_center_norm', 'bbox_y_center_norm', 'bbox_width_norm', 'bbox_height_norm', 'alignment_review_status']

KIỂM TRA FILE ĐẦU VÀO


,data_file,file_path,exists,file_size_mb
0,image_manifest,C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb06...,True,0.523
1,detection_annotations,C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb06...,True,0.673
2,complete_dataset,C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb06...,True,1.306



TỔNG QUAN DETECTION DATASET


,component,total_records,unique_ids
0,Image Manifest,1592,1592
1,Detection Annotations,1696,1696
2,Patient,892,892



PHÂN BỐ DATASET SPLIT


,dataset_split,annotation_count,image_count,patient_count
0,test,378,361,201
1,train,1054,982,552
2,validation,264,249,139


... (13 outputs left out)

,label,present_in_dataset
0,BENIGN,True
1,BENIGN_WITHOUT_CALLBACK,True
2,MALIGNANT,True



KIỂM TRA TÍNH TOÀN VẸN ĐẦU VÀO


,metric,value,expected,status
0,unique_images,1592,1592,PASS
1,unique_annotations,1696,1696,PASS
2,unique_patients,892,892,PASS
3,duplicate_image_ids,0,0,PASS
4,duplicate_annotation_ids,0,0,PASS
5,missing_image_mapping,0,0,PASS
6,unannotated_images,0,0,PASS
7,image_path_exists,1592,1592,PASS
8,patient_split_leakage,0,0,PASS
9,valid_bounding_boxes,1696,1696,PASS



TỔNG PASS: 12
TỔNG CHECK: 0

ĐÃ LƯU KẾT QUẢ TẠI:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design


### Thống kê đặc trưng kỹ thuật và ý nghĩa thiết kế của Detection Dataset

| Đặc điểm | Kết quả | Ý nghĩa thiết kế |
| :--- | :---: | :--- |
| **Ảnh Full Image** | 1.592 ảnh | Đóng vai trò là dữ liệu đầu vào cho mô hình phát hiện tổn thương (*Object Detection*). |
| **Annotation** | 1.696 | Phản ánh thực tế có những ảnh chứa nhiều annotation (nhiều tổn thương trên cùng một ảnh). |
| **Kích thước ảnh trung vị** | $5.281 \times 3.000\text{ px}$ | Thuộc nhóm ảnh y tế độ phân giải cao, đòi hỏi chiến lược quản lý bộ nhớ và resize phù hợp. |
| **BBox trung vị** | $304,5 \times 302\text{ px}$ | Cho thấy vùng tổn thương có kích thước tương đối nhỏ so với tổng thể toàn bộ khung ảnh. |
| **BBox nhỏ nhất** | $55 \times 86\text{ px}$ | Cần đặc biệt chú ý nguy cơ mất chi tiết hoặc mờ nhòe thông tin khi resize ảnh về kích thước nhỏ. |
| **BBox lớn nhất** | $1.365 \times 1.329\text{ px}$ | Minh chứng cho việc kích thước tổn thương biến thiên đáng kể trong dataset. |
| **Patient Leakage** | **0** | Tuyệt đối không phát hiện bệnh nhân xuất hiện ở nhiều tập khác nhau, đảm bảo tính độc lập giữa Train/Val/Test. |

In [2]:
# 07.2 — TỔNG QUAN NGHIÊN CỨU LIÊN QUAN

# 1. Cấu hình
PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

NB07_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb07_model_design"
)

NB07_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Tổng hợp tài liệu nghiên cứu

literature_records = [
    {
        "literature_id": "L01",
        "research_group": "Dataset Foundation",
        "authors": "Lee et al.",
        "year": 2017,
        "title": "A curated mammography data set for use in computer-aided detection and diagnosis research",
        "journal": "Scientific Data",
        "doi": "10.1038/sdata.2017.177",
        "dataset": "CBIS-DDSM",
        "data_scope": "Mass and Calcification",
        "research_task": "Dataset construction and annotation",
        "preprocessing": "Curated mammography images, ROI annotations, pathology labels and bounding-box information",
        "architecture": "Not a model-comparison study",
        "reported_result": "Introduced a standardized mammography dataset for CAD research",
        "limitations": "ROI annotations represent general lesion locations and should not automatically be interpreted as precise pixel-level segmentation",
        "relevance": "Dataset origin, annotation structure and ground-truth interpretation",
        "reference_url": "https://doi.org/10.1038/sdata.2017.177"
    },
    {
        "literature_id": "L02",
        "research_group": "Mass Detection and Classification",
        "authors": "Agarwal et al.",
        "year": 2019,
        "title": "Automatic mass detection in mammograms using deep convolutional neural networks",
        "journal": "Journal of Medical Imaging",
        "doi": "10.1117/1.JMI.6.3.031409",
        "dataset": "CBIS-DDSM and INbreast",
        "data_scope": "Mammography mass and non-mass regions",
        "research_task": "Mass detection and patch classification",
        "preprocessing": "Patch extraction and sliding-window approach",
        "architecture": "VGG16, ResNet50 and InceptionV3",
        "reported_result": "InceptionV3 reported 84.16% accuracy and Az 0.93 for mass/non-mass patch classification in the reported experiment",
        "limitations": "Mass/non-mass classification is different from benign/malignant pathology classification; results from different datasets and experiments must be distinguished",
        "relevance": "Provides architectural candidates and patch-based processing references",
        "reference_url": "https://doi.org/10.1117/1.JMI.6.3.031409"
    },
    {
        "literature_id": "L03",
        "research_group": "High-resolution Mammography",
        "authors": "Deep Learning to Improve Breast Cancer Detection on Screening Mammography",
        "year": 2019,
        "title": "Deep Learning to Improve Breast Cancer Detection on Screening Mammography",
        "journal": "Research publication",
        "doi": "",
        "dataset": "Screening mammography",
        "data_scope": "High-resolution mammographic images",
        "research_task": "Breast cancer detection",
        "preprocessing": "Image downsampling and patch-based processing",
        "architecture": "Deep convolutional neural networks",
        "reported_result": "Investigated the use of image patches and resolution reduction for mammography analysis",
        "limitations": "Dataset and evaluation setting differ from the current CBIS-DDSM Mass experiment",
        "relevance": "Background for image resolution, patch extraction and computational constraints",
        "reference_url": ""
    },
    {
        "literature_id": "L04",
        "research_group": "Mass Classification",
        "authors": "Quintana et al.",
        "year": 2023,
        "title": "Exploiting Patch Sizes and Resolutions for Multi-Scale Deep Learning in Mammogram Image Classification",
        "journal": "Bioengineering",
        "doi": "10.3390/bioengineering10050534",
        "dataset": "CBIS-DDSM",
        "data_scope": "Mammography mass and calcification cases",
        "research_task": "Multi-scale mammography classification",
        "preprocessing": "Comparison of different patch sizes and image resolutions",
        "architecture": "Multi-scale deep learning",
        "reported_result": "Reported AUC of 0.809 for a multi-scale configuration on CBIS-DDSM",
        "limitations": "Classification results do not directly establish the effectiveness of a tiling strategy for object detection",
        "relevance": "Supports investigation of patch size, resolution and multi-scale inputs",
        "reference_url": "https://doi.org/10.3390/bioengineering10050534"
    },
    {
        "literature_id": "L05",
        "research_group": "Mass Classification",
        "authors": "Liao and Aagaard",
        "year": 2024,
        "title": "An open codebase for enhancing transparency in deep learning-based breast cancer diagnosis utilizing CBIS-DDSM data",
        "journal": "Scientific Reports",
        "doi": "10.1038/s41598-024-78648-0",
        "dataset": "CBIS-DDSM",
        "data_scope": "Mass subset",
        "research_task": "Breast mass pathology classification",
        "preprocessing": "Full Image and ROI-based image processing",
        "architecture": "ResNet50 and Xception",
        "reported_result": "Preserved original pathology information and derived binary labels by mapping BENIGN and BENIGN_WITHOUT_CALLBACK to the benign class and MALIGNANT to the malignant class",
        "limitations": "Classification setting and data split may differ from the current experiment",
        "relevance": "Direct reference for binary pathology label mapping and reproducible CBIS-DDSM experiments",
        "reference_url": "https://doi.org/10.1038/s41598-024-78648-0"
    },
    {
        "literature_id": "L06",
        "research_group": "Mass Detection and Localization",
        "authors": "Manolakis et al.",
        "year": 2025,
        "title": "A Two-Stage Lightweight Deep Learning Framework for Mass Detection and Segmentation in Mammograms Using YOLOv5 and Depthwise SegNet",
        "journal": "Journal of Imaging Informatics in Medicine",
        "doi": "10.1007/s10278-025-01471-0",
        "dataset": "CBIS-DDSM and INbreast",
        "data_scope": "Mammography Mass",
        "research_task": "Mass detection and segmentation",
        "preprocessing": "Image resizing, padding and detection-region processing",
        "architecture": "YOLOv5n and Depthwise SegNet; YOLOv8n also evaluated",
        "reported_result": "Reported CBIS-DDSM mAP@50 of 50.3% for YOLOv5n and 54.0% for YOLOv8n in the reported experiment",
        "limitations": "Detection and segmentation metrics must be evaluated separately; hardware and dataset protocols differ from the current experiment",
        "relevance": "Direct reference for YOLO-based Mass detection and two-stage system design",
        "reference_url": "https://doi.org/10.1007/s10278-025-01471-0"
    }
]

literature_df = pd.DataFrame(literature_records)

# 3. Bảng phân nhóm nghiên cứu

research_group_summary_df = (
    literature_df.groupby("research_group")
    .agg(
        study_count=("literature_id", "count"),
        publication_year_min=("year", "min"),
        publication_year_max=("year", "max"),
        study_ids=("literature_id", lambda x: ", ".join(x))
    )
    .reset_index()
)

# 4. Bảng tổng hợp kiến trúc mô hình

architecture_records = [
    {
        "architecture": "YOLOv5n",
        "model_family": "Object Detection",
        "intended_role": "Mass Detection and Localization",
        "reference_studies": "L06",
        "comparison_task": "Bounding-box detection",
        "design_consideration": "Lightweight detector; investigate high-resolution input and small-object handling"
    },
    {
        "architecture": "VGG16",
        "model_family": "CNN Classification",
        "intended_role": "Pathology Classification",
        "reference_studies": "L02",
        "comparison_task": "Mass region classification",
        "design_consideration": "Baseline CNN with relatively large parameter count"
    },
    {
        "architecture": "ResNet50",
        "model_family": "CNN Classification",
        "intended_role": "Pathology Classification",
        "reference_studies": "L02; L05",
        "comparison_task": "Mass region classification",
        "design_consideration": "Residual connections and transfer-learning suitability"
    },
    {
        "architecture": "InceptionV3",
        "model_family": "CNN Classification",
        "intended_role": "Pathology Classification",
        "reference_studies": "L02",
        "comparison_task": "Mass region classification",
        "design_consideration": "Multi-scale feature extraction through inception modules"
    }
]

architecture_reference_df = pd.DataFrame(architecture_records)

# 5. Bảng tổng hợp chiến lược xử lý ảnh

preprocessing_records = [
    {
        "method": "Whole-image resizing",
        "reference_studies": "L03; L06",
        "purpose": "Reduce input dimensions and computational cost",
        "potential_advantage": "Simple pipeline and global image context",
        "potential_limitation": "Small lesions may lose spatial detail"
    },
    {
        "method": "Patch-based processing",
        "reference_studies": "L02; L03; L04",
        "purpose": "Analyze local mammographic regions",
        "potential_advantage": "Preserve more local image detail",
        "potential_limitation": "May lose global context and increase sample count"
    },
    {
        "method": "Multi-scale processing",
        "reference_studies": "L04",
        "purpose": "Combine information from different image scales",
        "potential_advantage": "Represent lesions at different spatial scales",
        "potential_limitation": "More complex data preparation and model design"
    },
    {
        "method": "Full Image and ROI processing",
        "reference_studies": "L05",
        "purpose": "Use global mammography context and local lesion regions",
        "potential_advantage": "Supports pathology classification using lesion-focused inputs",
        "potential_limitation": "ROI quality and image-to-ROI correspondence affect results"
    },
    {
        "method": "Resize with padding",
        "reference_studies": "L06",
        "purpose": "Preserve image aspect ratio when preparing detector inputs",
        "potential_advantage": "Avoid geometric distortion",
        "potential_limitation": "Image content is still downsampled"
    }
]

preprocessing_reference_df = pd.DataFrame(preprocessing_records)

# 6. Bảng tổng hợp chiến lược nhãn Pathology

pathology_mapping_df = pd.DataFrame([
    {
        "original_pathology": "BENIGN",
        "binary_pathology": "BENIGN",
        "mapping_rule": "Preserve as benign"
    },
    {
        "original_pathology": "BENIGN_WITHOUT_CALLBACK",
        "binary_pathology": "BENIGN",
        "mapping_rule": "Merge into benign for binary experiment"
    },
    {
        "original_pathology": "MALIGNANT",
        "binary_pathology": "MALIGNANT",
        "mapping_rule": "Preserve as malignant"
    }
])

# 7. Bảng lưu ý khi đối chiếu nghiên cứu

comparison_guidelines_df = pd.DataFrame([
    {
        "comparison_dimension": "Dataset",
        "consideration": "Distinguish CBIS-DDSM, INbreast and other mammography datasets"
    },
    {
        "comparison_dimension": "Data unit",
        "consideration": "Distinguish images, patients, lesions, annotations and patches"
    },
    {
        "comparison_dimension": "Research task",
        "consideration": "Do not compare detection, segmentation and classification metrics directly"
    },
    {
        "comparison_dimension": "Ground truth",
        "consideration": "Check whether labels originate from pathology, bounding boxes, ROI masks or radiologist annotations"
    },
    {
        "comparison_dimension": "Data split",
        "consideration": "Check patient-level separation and train, validation and test protocols"
    },
    {
        "comparison_dimension": "Preprocessing",
        "consideration": "Compare image resolution, resizing, cropping, patch extraction and normalization"
    },
    {
        "comparison_dimension": "Evaluation metrics",
        "consideration": "Compare the same metric and evaluation protocol before interpreting performance differences"
    },
    {
        "comparison_dimension": "Hardware",
        "consideration": "Consider GPU memory, batch size, input resolution and computational cost"
    }
])

# 8. Xuất các bảng tổng hợp

output_tables = {
    "nb07_2_literature_review.csv": literature_df,
    "nb07_2_research_group_summary.csv": research_group_summary_df,
    "nb07_2_architecture_reference.csv": architecture_reference_df,
    "nb07_2_preprocessing_reference.csv": preprocessing_reference_df,
    "nb07_2_pathology_mapping_reference.csv": pathology_mapping_df,
    "nb07_2_comparison_guidelines.csv": comparison_guidelines_df
}

for filename, dataframe in output_tables.items():
    dataframe.to_csv(
        NB07_OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 9. Kiểm tra kết quả

print("TỔNG QUAN TÀI LIỆU")
print("Số nghiên cứu:", len(literature_df))
print("Số nhóm nghiên cứu:", literature_df["research_group"].nunique())
display(
    literature_df[
        [
            "literature_id",
            "authors",
            "year",
            "research_group",
            "research_task",
            "architecture"
        ]
    ]
)

print("\nPHÂN NHÓM NGHIÊN CỨU")
display(research_group_summary_df)

print("\nTỔNG HỢP KIẾN TRÚC")
display(architecture_reference_df)

print("\nTỔNG HỢP PHƯƠNG PHÁP XỬ LÝ ẢNH")
display(preprocessing_reference_df)

print("\nÁNH XẠ NHÃN PATHOLOGY")
display(pathology_mapping_df)

print("\nNGUYÊN TẮC ĐỐI CHIẾU NGHIÊN CỨU")
display(comparison_guidelines_df)

print("\nĐÃ LƯU CÁC BẢNG TẠI:")
print(NB07_OUTPUT_DIR)


TỔNG QUAN TÀI LIỆU
Số nghiên cứu: 6
Số nhóm nghiên cứu: 5


,literature_id,authors,year,research_group,research_task,architecture
0,L01,Lee et al.,2017,Dataset Foundation,Dataset construction and annotation,Not a model-comparison study
1,L02,Agarwal et al.,2019,Mass Detection and Classification,Mass detection and patch classification,"VGG16, ResNet50 and InceptionV3"
2,L03,Deep Learning to Improve Breast Cancer Detecti...,2019,High-resolution Mammography,Breast cancer detection,Deep convolutional neural networks
3,L04,Quintana et al.,2023,Mass Classification,Multi-scale mammography classification,Multi-scale deep learning
4,L05,Liao and Aagaard,2024,Mass Classification,Breast mass pathology classification,ResNet50 and Xception
5,L06,Manolakis et al.,2025,Mass Detection and Localization,Mass detection and segmentation,YOLOv5n and Depthwise SegNet; YOLOv8n also eva...



PHÂN NHÓM NGHIÊN CỨU


,research_group,study_count,publication_year_min,publication_year_max,study_ids
0,Dataset Foundation,1,2017,2017,L01
1,High-resolution Mammography,1,2019,2019,L03
2,Mass Classification,2,2023,2024,"L04, L05"
3,Mass Detection and Classification,1,2019,2019,L02
4,Mass Detection and Localization,1,2025,2025,L06



TỔNG HỢP KIẾN TRÚC


,architecture,model_family,intended_role,reference_studies,comparison_task,design_consideration
0,YOLOv5n,Object Detection,Mass Detection and Localization,L06,Bounding-box detection,Lightweight detector; investigate high-resolut...
1,VGG16,CNN Classification,Pathology Classification,L02,Mass region classification,Baseline CNN with relatively large parameter c...
2,ResNet50,CNN Classification,Pathology Classification,L02; L05,Mass region classification,Residual connections and transfer-learning sui...
3,InceptionV3,CNN Classification,Pathology Classification,L02,Mass region classification,Multi-scale feature extraction through incepti...


... (3 outputs left out)

,original_pathology,binary_pathology,mapping_rule
0,BENIGN,BENIGN,Preserve as benign
1,BENIGN_WITHOUT_CALLBACK,BENIGN,Merge into benign for binary experiment
2,MALIGNANT,MALIGNANT,Preserve as malignant



NGUYÊN TẮC ĐỐI CHIẾU NGHIÊN CỨU


,comparison_dimension,consideration
0,Dataset,"Distinguish CBIS-DDSM, INbreast and other mamm..."
1,Data unit,"Distinguish images, patients, lesions, annotat..."
2,Research task,"Do not compare detection, segmentation and cla..."
3,Ground truth,"Check whether labels originate from pathology,..."
4,Data split,"Check patient-level separation and train, vali..."
5,Preprocessing,"Compare image resolution, resizing, cropping, ..."
6,Evaluation metrics,Compare the same metric and evaluation protoco...
7,Hardware,"Consider GPU memory, batch size, input resolut..."



ĐÃ LƯU CÁC BẢNG TẠI:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design


In [4]:
# 07.3 — SO SÁNH KIẾN TRÚC MÔ HÌNH

# 1. Cấu hình

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

NB07_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb07_model_design"
)

NB07_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Tổng quan kiến trúc và nhiệm vụ

architecture_records = [
    {
        "model": "YOLOv5n",
        "stage": "Stage 1",
        "task": "Mass Detection and Localization",
        "model_family": "One-stage Object Detector",
        "feature_extraction": "CSP-based backbone and PANet-style feature aggregation",
        "prediction_output": "Bounding Box, Objectness and Class Confidence",
        "main_strength": "Lightweight architecture and end-to-end bounding-box prediction",
        "main_limitation": "Small lesion detection may be affected by input resolution and feature-map stride",
        "hardware_consideration": "Relatively lightweight, but high-resolution input increases VRAM requirements",
        "evaluation_metrics": "Precision, Recall, IoU, mAP@50, mAP@50-95",
        "reference_studies": "L06"
    },
    {
        "model": "VGG16",
        "stage": "Stage 2",
        "task": "Binary Pathology Classification",
        "model_family": "Convolutional Neural Network",
        "feature_extraction": "Sequential convolutional blocks with 3x3 filters",
        "prediction_output": "BENIGN and MALIGNANT probabilities",
        "main_strength": "Simple and well-established architecture for transfer learning",
        "main_limitation": "Large parameter count and computational cost in fully connected layers",
        "hardware_consideration": "Requires attention to batch size and input dimensions",
        "evaluation_metrics": "Accuracy, Precision, Recall, F1-score, ROC-AUC and Confusion Matrix",
        "reference_studies": "L02"
    },
    {
        "model": "ResNet50",
        "stage": "Stage 2",
        "task": "Binary Pathology Classification",
        "model_family": "Residual Convolutional Neural Network",
        "feature_extraction": "Residual blocks with skip connections",
        "prediction_output": "BENIGN and MALIGNANT probabilities",
        "main_strength": "Residual learning supports deeper feature extraction",
        "main_limitation": "Deeper network may increase computational requirements",
        "hardware_consideration": "Batch size and input resolution need to be evaluated on 6 GB VRAM",
        "evaluation_metrics": "Accuracy, Precision, Recall, F1-score, ROC-AUC and Confusion Matrix",
        "reference_studies": "L02; L05"
    },
    {
        "model": "InceptionV3",
        "stage": "Stage 2",
        "task": "Binary Pathology Classification",
        "model_family": "Multi-branch Convolutional Neural Network",
        "feature_extraction": "Inception modules with parallel convolutional branches",
        "prediction_output": "BENIGN and MALIGNANT probabilities",
        "main_strength": "Extracts features at multiple receptive-field scales",
        "main_limitation": "More complex architecture and feature-flow structure",
        "hardware_consideration": "Input dimensions and batch size should follow the selected implementation",
        "evaluation_metrics": "Accuracy, Precision, Recall, F1-score, ROC-AUC and Confusion Matrix",
        "reference_studies": "L02"
    }
]

architecture_comparison_df = pd.DataFrame(architecture_records)

# 3. So sánh vai trò giữa hai giai đoạn

pipeline_records = [
    {
        "stage": "Stage 1",
        "component": "YOLOv5n",
        "input": "Preprocessed Full Image or image patches",
        "target": "Mass",
        "output": "Predicted Bounding Boxes and Confidence Scores",
        "purpose": "Identify and localize suspicious mass regions"
    },
    {
        "stage": "Stage 2",
        "component": "VGG16 / ResNet50 / InceptionV3",
        "input": "Mass crops generated from predicted bounding boxes",
        "target": "Binary Pathology",
        "output": "BENIGN or MALIGNANT probabilities",
        "purpose": "Classify the detected mass region"
    }
]

pipeline_design_df = pd.DataFrame(pipeline_records)

# 4. So sánh đặc điểm kiến trúc

architecture_characteristics_records = [
    {
        "model": "YOLOv5n",
        "architecture_characteristic": "Single-stage detection",
        "feature_strategy": "Multi-scale feature maps",
        "spatial_output": "Bounding boxes",
        "classification_output": "Detection class confidence",
        "primary_design_factor": "Small-object detection and localization"
    },
    {
        "model": "VGG16",
        "architecture_characteristic": "Sequential CNN",
        "feature_strategy": "Stacked convolutional layers",
        "spatial_output": "Not directly provided",
        "classification_output": "Binary class probabilities",
        "primary_design_factor": "Baseline classification performance and model complexity"
    },
    {
        "model": "ResNet50",
        "architecture_characteristic": "Residual CNN",
        "feature_strategy": "Residual feature learning",
        "spatial_output": "Not directly provided",
        "classification_output": "Binary class probabilities",
        "primary_design_factor": "Feature learning and transfer-learning performance"
    },
    {
        "model": "InceptionV3",
        "architecture_characteristic": "Multi-branch CNN",
        "feature_strategy": "Parallel convolutional filters",
        "spatial_output": "Not directly provided",
        "classification_output": "Binary class probabilities",
        "primary_design_factor": "Multi-scale feature representation"
    }
]

architecture_characteristics_df = pd.DataFrame(
    architecture_characteristics_records
)

# 5. Tiêu chí đánh giá Detection

detection_evaluation_df = pd.DataFrame([
    {
        "metric": "Precision",
        "purpose": "Measure the proportion of predicted detections that are correct",
        "interpretation": "Higher precision indicates fewer false-positive detections"
    },
    {
        "metric": "Recall",
        "purpose": "Measure the proportion of ground-truth lesions detected",
        "interpretation": "Higher recall indicates fewer missed lesions"
    },
    {
        "metric": "IoU",
        "purpose": "Measure overlap between predicted and ground-truth bounding boxes",
        "interpretation": "Higher IoU indicates better localization"
    },
    {
        "metric": "mAP@50",
        "purpose": "Mean Average Precision at IoU threshold 0.50",
        "interpretation": "Summarizes detection precision across confidence thresholds"
    },
    {
        "metric": "mAP@50-95",
        "purpose": "Average precision across IoU thresholds from 0.50 to 0.95",
        "interpretation": "Evaluates detection and localization under stricter overlap requirements"
    }
])

# 6. Tiêu chí đánh giá Classification

classification_evaluation_df = pd.DataFrame([
    {
        "metric": "Accuracy",
        "purpose": "Measure overall classification correctness",
        "interpretation": "Should be considered together with class distribution"
    },
    {
        "metric": "Precision",
        "purpose": "Measure the correctness of positive predictions",
        "interpretation": "For MALIGNANT, measures the proportion of malignant predictions that are truly malignant"
    },
    {
        "metric": "Recall",
        "purpose": "Measure the proportion of actual positive cases identified",
        "interpretation": "For MALIGNANT, measures the proportion of malignant cases correctly identified"
    },
    {
        "metric": "F1-score",
        "purpose": "Harmonic mean of Precision and Recall",
        "interpretation": "Balances precision and recall"
    },
    {
        "metric": "ROC-AUC",
        "purpose": "Measure ranking ability across classification thresholds",
        "interpretation": "Evaluates separation between BENIGN and MALIGNANT"
    },
    {
        "metric": "Confusion Matrix",
        "purpose": "Summarize actual and predicted class counts",
        "interpretation": "Shows class-specific errors"
    }
])

# 7. Các yếu tố cần kiểm soát khi so sánh mô hình

experimental_control_df = pd.DataFrame([
    {
        "factor": "Dataset split",
        "control": "Use the same patient-level Train, Validation and Test split"
    },
    {
        "factor": "Input data",
        "control": "Use the same source Mass annotations and corresponding image regions"
    },
    {
        "factor": "Pathology labels",
        "control": "Preserve original labels and apply the same binary mapping"
    },
    {
        "factor": "Preprocessing",
        "control": "Apply consistent image normalization and resizing policies"
    },
    {
        "factor": "Data augmentation",
        "control": "Use equivalent augmentation principles across classifiers"
    },
    {
        "factor": "Training configuration",
        "control": "Document optimizer, learning rate, batch size, epochs and random seed"
    },
    {
        "factor": "Model selection",
        "control": "Use Validation data for model selection and keep Test data untouched until final evaluation"
    },
    {
        "factor": "Hardware",
        "control": "Record GPU memory, training time and inference time"
    }
])

# 8. Định hướng lựa chọn mô hình

model_selection_records = [
    {
        "model": "YOLOv5n",
        "decision_status": "Primary detection candidate",
        "selection_reason": "Directly addresses bounding-box detection and has a CBIS-DDSM research reference",
        "remaining_question": "Which image resolution and preprocessing strategy provide the best balance between localization and GPU memory?"
    },
    {
        "model": "VGG16",
        "decision_status": "Classification comparison candidate",
        "selection_reason": "Provides a conventional CNN baseline",
        "remaining_question": "Can its classification performance justify its parameter and memory requirements?"
    },
    {
        "model": "ResNet50",
        "decision_status": "Classification comparison candidate",
        "selection_reason": "Residual architecture with prior CBIS-DDSM classification use",
        "remaining_question": "Does residual feature learning improve pathology classification on the current dataset?"
    },
    {
        "model": "InceptionV3",
        "decision_status": "Classification comparison candidate",
        "selection_reason": "Provides a multi-branch feature extraction approach",
        "remaining_question": "Does multi-scale feature extraction improve discrimination between benign and malignant masses?"
    }
]

model_selection_df = pd.DataFrame(model_selection_records)

# 9. Kiểm tra cấu trúc so sánh

architecture_integrity_df = pd.DataFrame([
    {
        "check": "Number of candidate architectures",
        "value": architecture_comparison_df["model"].nunique(),
        "expected": 4
    },
    {
        "check": "Number of detection candidates",
        "value": architecture_comparison_df.loc[
            architecture_comparison_df["stage"] == "Stage 1",
            "model"
        ].nunique(),
        "expected": 1
    },
    {
        "check": "Number of classification candidates",
        "value": architecture_comparison_df.loc[
            architecture_comparison_df["stage"] == "Stage 2",
            "model"
        ].nunique(),
        "expected": 3
    },
    {
        "check": "Number of pipeline stages",
        "value": pipeline_design_df["stage"].nunique(),
        "expected": 2
    }
])

architecture_integrity_df["status"] = (
    architecture_integrity_df["value"]
    .eq(architecture_integrity_df["expected"])
    .map({True: "PASS", False: "CHECK"})
)

# 10. Xuất kết quả

output_tables = {
    "nb07_3_architecture_comparison.csv": architecture_comparison_df,
    "nb07_3_two_stage_pipeline_design.csv": pipeline_design_df,
    "nb07_3_architecture_characteristics.csv": architecture_characteristics_df,
    "nb07_3_detection_evaluation_metrics.csv": detection_evaluation_df,
    "nb07_3_classification_evaluation_metrics.csv": classification_evaluation_df,
    "nb07_3_experimental_controls.csv": experimental_control_df,
    "nb07_3_model_selection_candidates.csv": model_selection_df,
    "nb07_3_architecture_integrity.csv": architecture_integrity_df
}

for filename, dataframe in output_tables.items():
    dataframe.to_csv(
        NB07_OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 11. Hiển thị kết quả

print("SO SÁNH KIẾN TRÚC MÔ HÌNH")
display(architecture_comparison_df)

print("\nTHIẾT KẾ TWO-STAGE PIPELINE")
display(pipeline_design_df)

print("\nĐẶC ĐIỂM KIẾN TRÚC")
display(architecture_characteristics_df)

print("\nMETRICS — DETECTION")
display(detection_evaluation_df)

print("\nMETRICS — CLASSIFICATION")
display(classification_evaluation_df)

print("\nCÁC YẾU TỐ KIỂM SOÁT THỰC NGHIỆM")
display(experimental_control_df)

print("\nĐỊNH HƯỚNG LỰA CHỌN MÔ HÌNH")
display(model_selection_df)

print("\nKIỂM TRA TÍNH TOÀN VẸN")
display(architecture_integrity_df)

print(
    "\nTỔNG PASS:",
    int((architecture_integrity_df["status"] == "PASS").sum())
)

print(
    "TỔNG CHECK:",
    int((architecture_integrity_df["status"] == "CHECK").sum())
)

print("\nĐÃ LƯU KẾT QUẢ TẠI:")
print(NB07_OUTPUT_DIR)


SO SÁNH KIẾN TRÚC MÔ HÌNH


,model,stage,task,model_family,feature_extraction,prediction_output,main_strength,main_limitation,hardware_consideration,evaluation_metrics,reference_studies
0,YOLOv5n,Stage 1,Mass Detection and Localization,One-stage Object Detector,CSP-based backbone and PANet-style feature agg...,"Bounding Box, Objectness and Class Confidence",Lightweight architecture and end-to-end boundi...,Small lesion detection may be affected by inpu...,"Relatively lightweight, but high-resolution in...","Precision, Recall, IoU, mAP@50, mAP@50-95",L06
1,VGG16,Stage 2,Binary Pathology Classification,Convolutional Neural Network,Sequential convolutional blocks with 3x3 filters,BENIGN and MALIGNANT probabilities,Simple and well-established architecture for t...,Large parameter count and computational cost i...,Requires attention to batch size and input dim...,"Accuracy, Precision, Recall, F1-score, ROC-AUC...",L02
2,ResNet50,Stage 2,Binary Pathology Classification,Residual Convolutional Neural Network,Residual blocks with skip connections,BENIGN and MALIGNANT probabilities,Residual learning supports deeper feature extr...,Deeper network may increase computational requ...,Batch size and input resolution need to be eva...,"Accuracy, Precision, Recall, F1-score, ROC-AUC...",L02; L05
3,InceptionV3,Stage 2,Binary Pathology Classification,Multi-branch Convolutional Neural Network,Inception modules with parallel convolutional ...,BENIGN and MALIGNANT probabilities,Extracts features at multiple receptive-field ...,More complex architecture and feature-flow str...,Input dimensions and batch size should follow ...,"Accuracy, Precision, Recall, F1-score, ROC-AUC...",L02



THIẾT KẾ TWO-STAGE PIPELINE


,stage,component,input,target,output,purpose
0,Stage 1,YOLOv5n,Preprocessed Full Image or image patches,Mass,Predicted Bounding Boxes and Confidence Scores,Identify and localize suspicious mass regions
1,Stage 2,VGG16 / ResNet50 / InceptionV3,Mass crops generated from predicted bounding b...,Binary Pathology,BENIGN or MALIGNANT probabilities,Classify the detected mass region



ĐẶC ĐIỂM KIẾN TRÚC


,model,architecture_characteristic,feature_strategy,spatial_output,classification_output,primary_design_factor
0,YOLOv5n,Single-stage detection,Multi-scale feature maps,Bounding boxes,Detection class confidence,Small-object detection and localization
1,VGG16,Sequential CNN,Stacked convolutional layers,Not directly provided,Binary class probabilities,Baseline classification performance and model ...
2,ResNet50,Residual CNN,Residual feature learning,Not directly provided,Binary class probabilities,Feature learning and transfer-learning perform...
3,InceptionV3,Multi-branch CNN,Parallel convolutional filters,Not directly provided,Binary class probabilities,Multi-scale feature representation


... (7 outputs left out)

,model,decision_status,selection_reason,remaining_question
0,YOLOv5n,Primary detection candidate,Directly addresses bounding-box detection and ...,Which image resolution and preprocessing strat...
1,VGG16,Classification comparison candidate,Provides a conventional CNN baseline,Can its classification performance justify its...
2,ResNet50,Classification comparison candidate,Residual architecture with prior CBIS-DDSM cla...,Does residual feature learning improve patholo...
3,InceptionV3,Classification comparison candidate,Provides a multi-branch feature extraction app...,Does multi-scale feature extraction improve di...



KIỂM TRA TÍNH TOÀN VẸN


,check,value,expected,status
0,Number of candidate architectures,4,4,PASS
1,Number of detection candidates,1,1,PASS
2,Number of classification candidates,3,3,PASS
3,Number of pipeline stages,2,2,PASS



TỔNG PASS: 4
TỔNG CHECK: 0

ĐÃ LƯU KẾT QUẢ TẠI:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design


In [9]:
# 07.4 — PHÂN TÍCH ĐỘ PHÂN GIẢI VÀ CHIẾN LƯỢC CHIA ẢNH

import math
import pandas as pd
import numpy as np
from pathlib import Path

# 1. Cấu hình

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

NB06_7_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb06_detection_dataset"
    / "nb06_7_detection_dataset"
)

NB07_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb07_model_design"
)

NB07_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

IMAGE_MANIFEST_FILE = (
    NB06_7_OUTPUT_DIR
    / "nb06_7_detection_image_manifest.csv"
)

ANNOTATION_FILE = (
    NB06_7_OUTPUT_DIR
    / "nb06_7_detection_annotations.csv"
)

FULL_IMAGE_SIZES = [640, 1024, 1280]

PATCH_CONFIGS = [
    {"patch_size": 1024, "overlap": 0.20},
    {"patch_size": 1280, "overlap": 0.20}
]

# 2. Đọc dữ liệu

if not IMAGE_MANIFEST_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Image Manifest: {IMAGE_MANIFEST_FILE}"
    )

if not ANNOTATION_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Detection Annotations: {ANNOTATION_FILE}"
    )

resolution_images_df = pd.read_csv(IMAGE_MANIFEST_FILE)
resolution_annotations_df = pd.read_csv(ANNOTATION_FILE)

required_image_columns = [
    "image_id",
    "image_height",
    "image_width",
    "dataset_split"
]

required_annotation_columns = [
    "annotation_id",
    "image_id",
    "image_height",
    "image_width",
    "bbox_width",
    "bbox_height",
    "dataset_split",
    "pathology",
    "image_view"
]

missing_image_columns = [
    col for col in required_image_columns
    if col not in resolution_images_df.columns
]

missing_annotation_columns = [
    col for col in required_annotation_columns
    if col not in resolution_annotations_df.columns
]

if missing_image_columns:
    raise ValueError(
        f"Image Manifest thiếu cột: {missing_image_columns}"
    )

if missing_annotation_columns:
    raise ValueError(
        f"Detection Annotations thiếu cột: {missing_annotation_columns}"
    )

# 3. Kiểm tra và chuẩn hóa kiểu dữ liệu

for col in ["image_height", "image_width"]:
    resolution_images_df[col] = pd.to_numeric(
        resolution_images_df[col],
        errors="coerce"
    )

for col in [
    "image_height",
    "image_width",
    "bbox_width",
    "bbox_height"
]:
    resolution_annotations_df[col] = pd.to_numeric(
        resolution_annotations_df[col],
        errors="coerce"
    )

# Tính diện tích Bounding Box từ chiều rộng và chiều cao
resolution_annotations_df["bbox_area"] = (
    resolution_annotations_df["bbox_width"]
    * resolution_annotations_df["bbox_height"]
)

if resolution_images_df["image_id"].duplicated().any():
    raise ValueError("Image Manifest có image_id trùng lặp.")

if resolution_annotations_df["annotation_id"].duplicated().any():
    raise ValueError("Detection Annotations có annotation_id trùng lặp.")

# 4. Đối chiếu kích thước ảnh giữa hai nguồn

manifest_dimension_df = resolution_images_df[
    ["image_id", "image_height", "image_width"]
].copy()

manifest_dimension_df = manifest_dimension_df.rename(
    columns={
        "image_height": "manifest_image_height",
        "image_width": "manifest_image_width"
    }
)

resolution_analysis_df = resolution_annotations_df.merge(
    manifest_dimension_df,
    on="image_id",
    how="left",
    validate="many_to_one",
    indicator=True
)

resolution_analysis_df["dimension_match"] = (
    resolution_analysis_df["_merge"].eq("both")
    & resolution_analysis_df["image_height"].eq(
        resolution_analysis_df["manifest_image_height"]
    )
    & resolution_analysis_df["image_width"].eq(
        resolution_analysis_df["manifest_image_width"]
    )
)

dimension_mismatch_count = int(
    (~resolution_analysis_df["dimension_match"]).sum()
)

if dimension_mismatch_count > 0:
    raise ValueError(
        f"Có {dimension_mismatch_count} annotation không khớp "
        "kích thước ảnh giữa Detection Annotations và Image Manifest."
    )

resolution_analysis_df = resolution_analysis_df.drop(
    columns=["_merge"]
)

# 5. Tạo các biến kích thước ảnh và Bounding Box

resolution_analysis_df["longest_side"] = resolution_analysis_df[
    ["image_height", "image_width"]
].max(axis=1)

resolution_analysis_df["shortest_side"] = resolution_analysis_df[
    ["image_height", "image_width"]
].min(axis=1)

resolution_analysis_df["image_area"] = (
    resolution_analysis_df["image_height"]
    * resolution_analysis_df["image_width"]
)

resolution_analysis_df["input_data_status"] = np.where(
    resolution_analysis_df[
        [
            "image_height",
            "image_width",
            "bbox_width",
            "bbox_height"
        ]
    ].notna().all(axis=1)
    & (resolution_analysis_df["image_height"] > 0)
    & (resolution_analysis_df["image_width"] > 0)
    & (resolution_analysis_df["bbox_width"] > 0)
    & (resolution_analysis_df["bbox_height"] > 0),
    "valid",
    "invalid"
)

valid_resolution_df = resolution_analysis_df[
    resolution_analysis_df["input_data_status"] == "valid"
].copy()

if len(valid_resolution_df) != len(resolution_annotations_df):
    invalid_count = (
        len(resolution_annotations_df) - len(valid_resolution_df)
    )

    raise ValueError(
        f"Có {invalid_count} annotation thiếu kích thước ảnh "
        "hoặc Bounding Box không hợp lệ."
    )

# 6. Thống kê kích thước ảnh và Bounding Box gốc

image_dimension_summary_df = pd.DataFrame([
    {
        "metric": "image_height",
        "mean": resolution_images_df["image_height"].mean(),
        "std": resolution_images_df["image_height"].std(),
        "min": resolution_images_df["image_height"].min(),
        "p25": resolution_images_df["image_height"].quantile(0.25),
        "median": resolution_images_df["image_height"].median(),
        "p75": resolution_images_df["image_height"].quantile(0.75),
        "max": resolution_images_df["image_height"].max()
    },
    {
        "metric": "image_width",
        "mean": resolution_images_df["image_width"].mean(),
        "std": resolution_images_df["image_width"].std(),
        "min": resolution_images_df["image_width"].min(),
        "p25": resolution_images_df["image_width"].quantile(0.25),
        "median": resolution_images_df["image_width"].median(),
        "p75": resolution_images_df["image_width"].quantile(0.75),
        "max": resolution_images_df["image_width"].max()
    },
    {
        "metric": "bbox_width",
        "mean": valid_resolution_df["bbox_width"].mean(),
        "std": valid_resolution_df["bbox_width"].std(),
        "min": valid_resolution_df["bbox_width"].min(),
        "p25": valid_resolution_df["bbox_width"].quantile(0.25),
        "median": valid_resolution_df["bbox_width"].median(),
        "p75": valid_resolution_df["bbox_width"].quantile(0.75),
        "max": valid_resolution_df["bbox_width"].max()
    },
    {
        "metric": "bbox_height",
        "mean": valid_resolution_df["bbox_height"].mean(),
        "std": valid_resolution_df["bbox_height"].std(),
        "min": valid_resolution_df["bbox_height"].min(),
        "p25": valid_resolution_df["bbox_height"].quantile(0.25),
        "median": valid_resolution_df["bbox_height"].median(),
        "p75": valid_resolution_df["bbox_height"].quantile(0.75),
        "max": valid_resolution_df["bbox_height"].max()
    }
])

# 7. Ước lượng kích thước Bounding Box sau letterbox resize

full_image_resolution_records = []

for input_size in FULL_IMAGE_SIZES:
    scale = input_size / valid_resolution_df["longest_side"]

    resized_width = valid_resolution_df["bbox_width"] * scale
    resized_height = valid_resolution_df["bbox_height"] * scale

    resized_area = resized_width * resized_height

    full_image_resolution_records.append({
        "input_size": input_size,
        "annotation_count": len(valid_resolution_df),
        "scale_mean": scale.mean(),
        "bbox_width_mean": resized_width.mean(),
        "bbox_width_p10": resized_width.quantile(0.10),
        "bbox_width_median": resized_width.median(),
        "bbox_width_p90": resized_width.quantile(0.90),
        "bbox_width_min": resized_width.min(),
        "bbox_height_mean": resized_height.mean(),
        "bbox_height_p10": resized_height.quantile(0.10),
        "bbox_height_median": resized_height.median(),
        "bbox_height_p90": resized_height.quantile(0.90),
        "bbox_height_min": resized_height.min(),
        "bbox_area_median": resized_area.median(),
        "bbox_area_below_32_squared": int(
            (resized_area < 32 ** 2).sum()
        ),
        "bbox_area_below_32_squared_ratio": (
            resized_area < 32 ** 2
        ).mean(),
        "relative_input_pixels": (input_size / 640) ** 2
    })

full_image_resolution_comparison_df = pd.DataFrame(
    full_image_resolution_records
)

# 8. Thống kê theo từng tập dữ liệu

split_resolution_records = []

for split_name, split_df in valid_resolution_df.groupby("dataset_split"):
    for input_size in FULL_IMAGE_SIZES:
        scale = input_size / split_df["longest_side"]

        resized_width = split_df["bbox_width"] * scale
        resized_height = split_df["bbox_height"] * scale

        resized_area = resized_width * resized_height

        split_resolution_records.append({
            "dataset_split": split_name,
            "input_size": input_size,
            "annotation_count": len(split_df),
            "bbox_width_median": resized_width.median(),
            "bbox_height_median": resized_height.median(),
            "bbox_width_p10": resized_width.quantile(0.10),
            "bbox_height_p10": resized_height.quantile(0.10),
            "bbox_area_below_32_squared": int(
                (resized_area < 32 ** 2).sum()
            )
        })

split_resolution_comparison_df = pd.DataFrame(
    split_resolution_records
)

# 9. Ước lượng số patch khi chia ảnh

def calculate_patch_count(image_length, patch_size, overlap):
    stride = max(1, int(round(patch_size * (1 - overlap))))

    if image_length <= patch_size:
        return 1

    return math.ceil(
        (image_length - patch_size) / stride
    ) + 1


patch_strategy_records = []

for config in PATCH_CONFIGS:
    patch_size = config["patch_size"]
    overlap = config["overlap"]

    row_counts = resolution_images_df["image_height"].apply(
        lambda x: calculate_patch_count(
            int(x), patch_size, overlap
        )
    )

    column_counts = resolution_images_df["image_width"].apply(
        lambda x: calculate_patch_count(
            int(x), patch_size, overlap
        )
    )

    total_tiles = row_counts * column_counts

    patch_strategy_records.append({
        "patch_size": patch_size,
        "overlap_ratio": overlap,
        "stride": int(round(patch_size * (1 - overlap))),
        "image_count": len(resolution_images_df),
        "total_estimated_patches": int(total_tiles.sum()),
        "mean_patches_per_image": total_tiles.mean(),
        "median_patches_per_image": total_tiles.median(),
        "min_patches_per_image": total_tiles.min(),
        "max_patches_per_image": total_tiles.max(),
        "relative_patch_pixels_vs_640": (
            patch_size / 640
        ) ** 2
    })

patch_strategy_comparison_df = pd.DataFrame(
    patch_strategy_records
)

# 10. So sánh định hướng chiến lược

strategy_records = [
    {
        "strategy": "Full Image 640",
        "spatial_coverage": "Entire mammogram",
        "resolution_characteristic": "Lowest input resolution among candidates",
        "potential_advantage": "Lower computational demand and faster initial experiments",
        "potential_limitation": "Small Mass regions may occupy fewer pixels",
        "main_question": "Is the lesion detail sufficient for reliable localization?"
    },
    {
        "strategy": "Full Image 1024",
        "spatial_coverage": "Entire mammogram",
        "resolution_characteristic": "Higher spatial detail than 640",
        "potential_advantage": "Retains more lesion detail while preserving global breast context",
        "potential_limitation": "Higher GPU memory and computation requirements",
        "main_question": "Does improved spatial detail justify the additional computation?"
    },
    {
        "strategy": "Full Image 1280",
        "spatial_coverage": "Entire mammogram",
        "resolution_characteristic": "Highest full-image resolution candidate",
        "potential_advantage": "Preserves more fine-grained image information",
        "potential_limitation": "May exceed practical training memory or reduce batch size",
        "main_question": "Can the available GPU handle this resolution efficiently?"
    },
    {
        "strategy": "Patch 1024, overlap 20%",
        "spatial_coverage": "Local image regions",
        "resolution_characteristic": "Uses original image pixels within each patch",
        "potential_advantage": "Avoids reducing the entire mammogram to a small square input",
        "potential_limitation": "Requires patch-label mapping and handling lesions near patch boundaries",
        "main_question": "Can patch-based training improve small-lesion localization?"
    },
    {
        "strategy": "Patch 1280, overlap 20%",
        "spatial_coverage": "Local image regions",
        "resolution_characteristic": "Larger local context than 1024 patches",
        "potential_advantage": "Preserves local detail while retaining more surrounding tissue",
        "potential_limitation": "More pixels per patch and potentially greater memory demand",
        "main_question": "Does the additional context improve detection enough to justify its cost?"
    }
]

input_strategy_comparison_df = pd.DataFrame(strategy_records)

# 11. Kiểm tra tính toàn vẹn

resolution_integrity_records = [
    {
        "check": "Unique images",
        "value": resolution_images_df["image_id"].nunique(),
        "expected": 1592
    },
    {
        "check": "Unique annotations",
        "value": resolution_annotations_df["annotation_id"].nunique(),
        "expected": 1696
    },
    {
        "check": "Annotations with valid image dimensions and bounding boxes",
        "value": len(valid_resolution_df),
        "expected": 1696
    },
    {
        "check": "Image dimensions matched between two source files",
        "value": int(resolution_analysis_df["dimension_match"].sum()),
        "expected": 1696
    },
    {
        "check": "Full-image resolution candidates",
        "value": len(full_image_resolution_comparison_df),
        "expected": 3
    },
    {
        "check": "Patch strategy candidates",
        "value": len(patch_strategy_comparison_df),
        "expected": 2
    }
]

resolution_integrity_df = pd.DataFrame(
    resolution_integrity_records
)

resolution_integrity_df["status"] = (
    resolution_integrity_df["value"]
    .eq(resolution_integrity_df["expected"])
    .map({True: "PASS", False: "CHECK"})
)

# 12. Xuất kết quả

output_tables = {
    "nb07_4_image_dimension_bbox_summary.csv":
        image_dimension_summary_df,

    "nb07_4_full_image_resolution_comparison.csv":
        full_image_resolution_comparison_df,

    "nb07_4_split_resolution_comparison.csv":
        split_resolution_comparison_df,

    "nb07_4_patch_strategy_comparison.csv":
        patch_strategy_comparison_df,

    "nb07_4_input_strategy_comparison.csv":
        input_strategy_comparison_df,

    "nb07_4_resolution_integrity.csv":
        resolution_integrity_df
}

for filename, dataframe in output_tables.items():
    dataframe.to_csv(
        NB07_OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 13. Hiển thị kết quả

print("THỐNG KÊ KÍCH THƯỚC ẢNH VÀ BOUNDING BOX GỐC")
display(image_dimension_summary_df)

print("\nSO SÁNH RESIZE TOÀN ẢNH")
display(full_image_resolution_comparison_df)

print("\nSO SÁNH ĐỘ PHÂN GIẢI THEO DATASET SPLIT")
display(split_resolution_comparison_df)

print("\nSO SÁNH CHIẾN LƯỢC PATCH")
display(patch_strategy_comparison_df)

print("\nSO SÁNH CÁC CHIẾN LƯỢC INPUT")
display(input_strategy_comparison_df)

print("\nKIỂM TRA TÍNH TOÀN VẸN")
display(resolution_integrity_df)

print(
    "\nTỔNG PASS:",
    int((resolution_integrity_df["status"] == "PASS").sum())
)

print(
    "TỔNG CHECK:",
    int((resolution_integrity_df["status"] == "CHECK").sum())
)

print("\nĐÃ LƯU KẾT QUẢ TẠI:")
print(NB07_OUTPUT_DIR)


THỐNG KÊ KÍCH THƯỚC ẢNH VÀ BOUNDING BOX GỐC


,metric,mean,std,min,p25,median,p75,max
0,image_height,5295.331030,679.737343,3920,4696.0,5281.0,5800.00,6931
1,image_width,3133.636307,545.635186,1786,2742.0,3000.0,3541.00,5431
2,bbox_width,331.056604,136.065702,55,242.0,304.5,382.25,1365
3,bbox_height,327.885024,135.803199,86,238.0,302.0,381.00,1329



SO SÁNH RESIZE TOÀN ẢNH


,input_size,annotation_count,scale_mean,bbox_width_mean,bbox_width_p10,bbox_width_median,bbox_width_p90,bbox_width_min,bbox_height_mean,bbox_height_p10,bbox_height_median,bbox_height_p90,bbox_height_min,bbox_area_median,bbox_area_below_32_squared,bbox_area_below_32_squared_ratio,relative_input_pixels
0,640,1696,0.122803,40.639141,22.659279,37.519577,61.963054,7.394958,40.386957,22.358709,37.044319,63.019997,9.542302,1378.252742,576,0.339623,1.00
1,1024,1696,0.196485,65.022626,36.254847,60.031324,99.140886,11.831933,64.619132,35.773935,59.270911,100.831995,15.267684,3528.327018,78,0.045991,2.56
2,1280,1696,0.245606,81.278282,45.318559,75.039155,123.926108,14.789916,80.773915,44.717419,74.088639,126.039994,19.084605,5513.010966,19,0.011203,4.00



SO SÁNH ĐỘ PHÂN GIẢI THEO DATASET SPLIT


,dataset_split,input_size,annotation_count,bbox_width_median,bbox_height_median,bbox_width_p10,bbox_height_p10,bbox_area_below_32_squared
0,test,640,378,37.472227,36.544645,23.139234,22.772157,129
1,test,1024,378,59.955564,58.471431,37.022774,36.435451,21
2,test,1280,378,74.944455,73.089289,46.278468,45.544314,2
3,train,640,1054,37.558702,37.231117,22.847953,22.251356,351
4,train,1024,1054,60.093924,59.569787,36.556725,35.602170,50
5,train,1280,1054,75.117405,74.462234,45.695906,44.502712,16
6,validation,640,264,37.411881,36.141565,21.969336,21.015298,96
7,validation,1024,264,59.859009,57.826504,35.150937,33.624477,7
8,validation,1280,264,74.823762,72.283129,43.938671,42.030596,1


... (3 outputs left out)

,strategy,spatial_coverage,resolution_characteristic,potential_advantage,potential_limitation,main_question
0,Full Image 640,Entire mammogram,Lowest input resolution among candidates,Lower computational demand and faster initial ...,Small Mass regions may occupy fewer pixels,Is the lesion detail sufficient for reliable l...
1,Full Image 1024,Entire mammogram,Higher spatial detail than 640,Retains more lesion detail while preserving gl...,Higher GPU memory and computation requirements,Does improved spatial detail justify the addit...
2,Full Image 1280,Entire mammogram,Highest full-image resolution candidate,Preserves more fine-grained image information,May exceed practical training memory or reduce...,Can the available GPU handle this resolution e...
3,"Patch 1024, overlap 20%",Local image regions,Uses original image pixels within each patch,Avoids reducing the entire mammogram to a smal...,Requires patch-label mapping and handling lesi...,Can patch-based training improve small-lesion ...
4,"Patch 1280, overlap 20%",Local image regions,Larger local context than 1024 patches,Preserves local detail while retaining more su...,More pixels per patch and potentially greater ...,Does the additional context improve detection ...



KIỂM TRA TÍNH TOÀN VẸN


,check,value,expected,status
0,Unique images,1592,1592,PASS
1,Unique annotations,1696,1696,PASS
2,Annotations with valid image dimensions and bo...,1696,1696,PASS
3,Image dimensions matched between two source files,1696,1696,PASS
4,Full-image resolution candidates,3,3,PASS
5,Patch strategy candidates,2,2,PASS



TỔNG PASS: 6
TỔNG CHECK: 0

ĐÃ LƯU KẾT QUẢ TẠI:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design


In [11]:
# 07.5 — KIỂM TRA KHẢ NĂNG ĐÁP ỨNG PHẦN CỨNG

import sys
import platform
import subprocess
import importlib.util
from pathlib import Path

# 1. Cấu hình

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

NB07_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb07_model_design"
)

NB07_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Thông tin hệ thống

system_records = [
    {"component": "Operating system", "value": platform.platform()},
    {"component": "Python version", "value": sys.version.split()[0]},
    {"component": "Python executable", "value": sys.executable},
    {"component": "Processor", "value": platform.processor()},
    {"component": "Architecture", "value": platform.machine()}
]

# 3. Kiểm tra PyTorch và CUDA

torch_available = importlib.util.find_spec("torch") is not None

torch_version = None
cuda_available = False
cuda_version = None
gpu_name = None
gpu_total_memory_gb = None
gpu_free_memory_gb = None
gpu_allocated_memory_gb = None
gpu_reserved_memory_gb = None
gpu_error = None

if torch_available:
    try:
        import torch

        torch_version = torch.__version__
        cuda_available = torch.cuda.is_available()
        cuda_version = torch.version.cuda

        if cuda_available:
            gpu_name = torch.cuda.get_device_name(0)

            free_bytes, total_bytes = torch.cuda.mem_get_info(0)

            gpu_total_memory_gb = total_bytes / (1024 ** 3)
            gpu_free_memory_gb = free_bytes / (1024 ** 3)

            gpu_allocated_memory_gb = (
                torch.cuda.memory_allocated(0) / (1024 ** 3)
            )

            gpu_reserved_memory_gb = (
                torch.cuda.memory_reserved(0) / (1024 ** 3)
            )

    except Exception as error:
        gpu_error = str(error)

system_records.extend([
    {
        "component": "PyTorch installed",
        "value": torch_available
    },
    {
        "component": "PyTorch version",
        "value": torch_version
    },
    {
        "component": "CUDA available in PyTorch",
        "value": cuda_available
    },
    {
        "component": "CUDA version used by PyTorch",
        "value": cuda_version
    },
    {
        "component": "GPU name",
        "value": gpu_name
    },
    {
        "component": "GPU total memory (GB)",
        "value": gpu_total_memory_gb
    },
    {
        "component": "GPU free memory (GB)",
        "value": gpu_free_memory_gb
    },
    {
        "component": "GPU allocated memory (GB)",
        "value": gpu_allocated_memory_gb
    },
    {
        "component": "GPU reserved memory (GB)",
        "value": gpu_reserved_memory_gb
    },
    {
        "component": "GPU inspection error",
        "value": gpu_error
    }
])

system_info_df = pd.DataFrame(system_records)

# 4. Kiểm tra thông tin GPU bằng nvidia-smi

nvidia_smi_available = False
nvidia_smi_output = None
nvidia_smi_error = None

try:
    result = subprocess.run(
        [
            "nvidia-smi",
            "--query-gpu=name,memory.total,memory.used,memory.free,driver_version",
            "--format=csv,noheader"
        ],
        capture_output=True,
        text=True,
        timeout=15
    )

    if result.returncode == 0:
        nvidia_smi_available = True
        nvidia_smi_output = result.stdout.strip()
    else:
        nvidia_smi_error = result.stderr.strip()

except Exception as error:
    nvidia_smi_error = str(error)

gpu_command_df = pd.DataFrame([
    {
        "check": "nvidia-smi available",
        "value": nvidia_smi_available
    },
    {
        "check": "GPU information",
        "value": nvidia_smi_output
    },
    {
        "check": "nvidia-smi error",
        "value": nvidia_smi_error
    }
])

# 5. Ước lượng dung lượng tensor đầu vào

# Giả định ảnh grayscale được chuyển thành 3 kênh
# và tensor sử dụng float32.
INPUT_CHANNELS = 3
BYTES_PER_VALUE = 4

INPUT_SIZES = [640, 1024, 1280]
BATCH_SIZES = [1, 2, 4]

tensor_records = []

for input_size in INPUT_SIZES:
    for batch_size in BATCH_SIZES:

        tensor_bytes = (
            batch_size
            * INPUT_CHANNELS
            * input_size
            * input_size
            * BYTES_PER_VALUE
        )

        tensor_records.append({
            "input_size": input_size,
            "batch_size": batch_size,
            "channels": INPUT_CHANNELS,
            "dtype": "float32",
            "tensor_memory_mb": tensor_bytes / (1024 ** 2),
            "tensor_memory_gb": tensor_bytes / (1024 ** 3)
        })

tensor_memory_df = pd.DataFrame(tensor_records)

# 6. Tổng hợp mức tiêu thụ pixel tương đối

resolution_cost_records = []

for input_size in INPUT_SIZES:
    resolution_cost_records.append({
        "input_size": input_size,
        "pixels_per_image": input_size ** 2,
        "relative_pixels_vs_640": (input_size / 640) ** 2,
        "relative_computation_indicator": (
            input_size / 640
        ) ** 2,
        "estimated_tensor_mb_batch_1": (
            INPUT_CHANNELS
            * input_size
            * input_size
            * BYTES_PER_VALUE
            / (1024 ** 2)
        )
    })

resolution_cost_df = pd.DataFrame(
    resolution_cost_records
)

# 7. Đánh giá khả năng kiểm tra GPU

if not torch_available:
    gpu_status = "PYTORCH_NOT_INSTALLED"
elif gpu_error is not None:
    gpu_status = "GPU_INSPECTION_ERROR"
elif not cuda_available:
    gpu_status = "CUDA_NOT_AVAILABLE"
else:
    gpu_status = "GPU_AVAILABLE"

hardware_readiness_df = pd.DataFrame([
    {
        "check": "PyTorch availability",
        "value": torch_available,
        "status": "PASS" if torch_available else "CHECK"
    },
    {
        "check": "CUDA availability",
        "value": cuda_available,
        "status": "PASS" if cuda_available else "CHECK"
    },
    {
        "check": "GPU identified",
        "value": gpu_name is not None,
        "status": "PASS" if gpu_name is not None else "CHECK"
    },
    {
        "check": "GPU memory inspected",
        "value": gpu_total_memory_gb is not None,
        "status": "PASS" if gpu_total_memory_gb is not None else "CHECK"
    },
    {
        "check": "nvidia-smi available",
        "value": nvidia_smi_available,
        "status": "PASS" if nvidia_smi_available else "CHECK"
    }
])

# 8. Xuất kết quả

output_tables = {
    "nb07_5_system_information.csv": system_info_df,
    "nb07_5_gpu_command_information.csv": gpu_command_df,
    "nb07_5_tensor_memory_estimation.csv": tensor_memory_df,
    "nb07_5_resolution_cost_comparison.csv": resolution_cost_df,
    "nb07_5_hardware_readiness.csv": hardware_readiness_df
}

for filename, dataframe in output_tables.items():
    dataframe.to_csv(
        NB07_OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 9. Hiển thị kết quả

print("THÔNG TIN HỆ THỐNG")
display(system_info_df)

print("\nTHÔNG TIN GPU TỪ NVIDIA-SMI")
display(gpu_command_df)

print("\nƯỚC TÍNH DUNG LƯỢNG TENSOR ĐẦU VÀO")
display(tensor_memory_df)

print("\nSO SÁNH CHI PHÍ ĐẦU VÀO THEO ĐỘ PHÂN GIẢI")
display(resolution_cost_df)

print("\nKIỂM TRA KHẢ NĂNG PHẦN CỨNG")
display(hardware_readiness_df)

print("\nGPU STATUS:", gpu_status)

print("\nĐÃ LƯU KẾT QUẢ TẠI:")
print(NB07_OUTPUT_DIR)


THÔNG TIN HỆ THỐNG


,component,value
0,Operating system,Windows-10-10.0.26300-SP0
1,Python version,3.11.18
2,Python executable,c:\Users\HOANG YEN\miniconda3\envs\cbis-thesis...
3,Processor,"Intel64 Family 6 Model 186 Stepping 2, Genuine..."
4,Architecture,AMD64
5,PyTorch installed,True
6,PyTorch version,2.5.1+cu121
7,CUDA available in PyTorch,True
8,CUDA version used by PyTorch,12.1
9,GPU name,NVIDIA GeForce RTX 4050 Laptop GPU



THÔNG TIN GPU TỪ NVIDIA-SMI


,check,value
0,nvidia-smi available,True
1,GPU information,"NVIDIA GeForce RTX 4050 Laptop GPU, 6141 MiB, ..."
2,nvidia-smi error,None



ƯỚC TÍNH DUNG LƯỢNG TENSOR ĐẦU VÀO


,input_size,batch_size,channels,dtype,tensor_memory_mb,tensor_memory_gb
0,640,1,3,float32,4.6875,0.004578
1,640,2,3,float32,9.3750,0.009155
2,640,4,3,float32,18.7500,0.018311
3,1024,1,3,float32,12.0000,0.011719
4,1024,2,3,float32,24.0000,0.023438
5,1024,4,3,float32,48.0000,0.046875
6,1280,1,3,float32,18.7500,0.018311
7,1280,2,3,float32,37.5000,0.036621
8,1280,4,3,float32,75.0000,0.073242



SO SÁNH CHI PHÍ ĐẦU VÀO THEO ĐỘ PHÂN GIẢI


,input_size,pixels_per_image,relative_pixels_vs_640,relative_computation_indicator,estimated_tensor_mb_batch_1
0,640,409600,1.00,1.00,4.6875
1,1024,1048576,2.56,2.56,12.0000
2,1280,1638400,4.00,4.00,18.7500



KIỂM TRA KHẢ NĂNG PHẦN CỨNG


,check,value,status
0,PyTorch availability,True,PASS
1,CUDA availability,True,PASS
2,GPU identified,True,PASS
3,GPU memory inspected,True,PASS
4,nvidia-smi available,True,PASS



GPU STATUS: GPU_AVAILABLE

ĐÃ LƯU KẾT QUẢ TẠI:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design


In [12]:
# 07.6 — MA TRẬN THIẾT KẾ THỰC NGHIỆM ĐẦU VÀO VÀ NHÃN

# 1. Cấu hình

PROJECT_ROOT = Path(r"C:\Breast-Cancer-CBIS-DDSM")

NB06_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb06_detection_dataset"
    / "nb06_7_detection_dataset"
)

NB07_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Data"
    / "processed"
    / "nb07_model_design"
)

NB07_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ANNOTATION_FILE = (
    NB06_OUTPUT_DIR
    / "nb06_7_detection_annotations.csv"
)

# 2. Đọc Detection Annotations

if not ANNOTATION_FILE.exists():
    raise FileNotFoundError(
        f"Không tìm thấy Detection Annotations: {ANNOTATION_FILE}"
    )

experiment_annotations_df = pd.read_csv(ANNOTATION_FILE)

required_columns = [
    "annotation_id",
    "patient_id",
    "dataset_split",
    "image_view",
    "pathology",
    "x_min",
    "y_min",
    "x_max",
    "y_max"
]

missing_columns = [
    col for col in required_columns
    if col not in experiment_annotations_df.columns
]

if missing_columns:
    raise ValueError(
        f"Detection Annotations thiếu cột: {missing_columns}"
    )

# 3. Tạo nhãn nhị phân dẫn xuất

pathology_mapping = {
    "BENIGN": "BENIGN",
    "BENIGN_WITHOUT_CALLBACK": "BENIGN",
    "MALIGNANT": "MALIGNANT"
}

experiment_annotations_df["binary_pathology"] = (
    experiment_annotations_df["pathology"]
    .map(pathology_mapping)
)

if experiment_annotations_df["binary_pathology"].isna().any():
    unknown_labels = (
        experiment_annotations_df.loc[
            experiment_annotations_df["binary_pathology"].isna(),
            "pathology"
        ]
        .dropna()
        .unique()
        .tolist()
    )

    raise ValueError(
        f"Phát hiện pathology chưa được ánh xạ: {unknown_labels}"
    )

binary_label_id_mapping = {
    "BENIGN": 0,
    "MALIGNANT": 1
}

experiment_annotations_df["binary_pathology_label_id"] = (
    experiment_annotations_df["binary_pathology"]
    .map(binary_label_id_mapping)
)

# 4. Tổng hợp phân bố nhãn gốc và nhãn nhị phân

original_pathology_summary_df = (
    experiment_annotations_df
    .groupby(
        ["dataset_split", "pathology"],
        dropna=False
    )
    .agg(
        annotation_count=("annotation_id", "nunique"),
        patient_count=("patient_id", "nunique")
    )
    .reset_index()
)

binary_pathology_summary_df = (
    experiment_annotations_df
    .groupby(
        ["dataset_split", "binary_pathology"],
        dropna=False
    )
    .agg(
        annotation_count=("annotation_id", "nunique"),
        patient_count=("patient_id", "nunique")
    )
    .reset_index()
)

# 5. Ma trận chiến lược đầu vào

input_strategy_matrix_df = pd.DataFrame([
    {
        "strategy": "Full Image 640",
        "input_size": 640,
        "input_type": "Full Image",
        "role": "Baseline",
        "spatial_coverage": "Entire mammogram",
        "relative_pixels_vs_640": 1.00,
        "priority": "Required",
        "evaluation_purpose": "Baseline detection performance"
    },
    {
        "strategy": "Full Image 1024",
        "input_size": 1024,
        "input_type": "Full Image",
        "role": "High-resolution full-image",
        "spatial_coverage": "Entire mammogram",
        "relative_pixels_vs_640": 2.56,
        "priority": "High",
        "evaluation_purpose": "Evaluate improved lesion detail"
    },
    {
        "strategy": "Full Image 1280",
        "input_size": 1280,
        "input_type": "Full Image",
        "role": "High-resolution full-image",
        "spatial_coverage": "Entire mammogram",
        "relative_pixels_vs_640": 4.00,
        "priority": "Conditional",
        "evaluation_purpose": "Evaluate high-resolution feasibility"
    },
    {
        "strategy": "Patch 1024 overlap 20%",
        "input_size": 1024,
        "input_type": "Patch",
        "role": "Local-region detection",
        "spatial_coverage": "Local image regions",
        "relative_pixels_vs_640": 2.56,
        "priority": "High",
        "evaluation_purpose": "Evaluate local-detail preservation"
    },
    {
        "strategy": "Patch 1280 overlap 20%",
        "input_size": 1280,
        "input_type": "Patch",
        "role": "Local-region detection",
        "spatial_coverage": "Local image regions",
        "relative_pixels_vs_640": 4.00,
        "priority": "Conditional",
        "evaluation_purpose": "Evaluate larger local context"
    }
])

# 6. Ma trận vai trò hai giai đoạn

pipeline_design_df = pd.DataFrame([
    {
        "stage": "Stage 1",
        "task": "Mass Detection and Localization",
        "candidate_model": "YOLOv5n",
        "input": "Full Image or image patches",
        "target": "Mass bounding boxes",
        "output": "Bounding boxes and confidence scores",
        "evaluation_metrics": "Precision, Recall, IoU, mAP@50, mAP@50-95"
    },
    {
        "stage": "Stage 2",
        "task": "Binary Pathology Classification",
        "candidate_model": "VGG16",
        "input": "Mass crop from predicted bounding box",
        "target": "BENIGN / MALIGNANT",
        "output": "Class probabilities",
        "evaluation_metrics": "Accuracy, Precision, Recall, F1, ROC-AUC"
    },
    {
        "stage": "Stage 2",
        "task": "Binary Pathology Classification",
        "candidate_model": "ResNet50",
        "input": "Mass crop from predicted bounding box",
        "target": "BENIGN / MALIGNANT",
        "output": "Class probabilities",
        "evaluation_metrics": "Accuracy, Precision, Recall, F1, ROC-AUC"
    },
    {
        "stage": "Stage 2",
        "task": "Binary Pathology Classification",
        "candidate_model": "InceptionV3",
        "input": "Mass crop from predicted bounding box",
        "target": "BENIGN / MALIGNANT",
        "output": "Class probabilities",
        "evaluation_metrics": "Accuracy, Precision, Recall, F1, ROC-AUC"
    }
])

# 7. Kiểm tra tính toàn vẹn

expected_original_labels = {
    "BENIGN",
    "BENIGN_WITHOUT_CALLBACK",
    "MALIGNANT"
}

actual_original_labels = set(
    experiment_annotations_df["pathology"].dropna().unique()
)

expected_binary_labels = {
    "BENIGN",
    "MALIGNANT"
}

actual_binary_labels = set(
    experiment_annotations_df["binary_pathology"].dropna().unique()
)

experiment_integrity_df = pd.DataFrame([
    {
        "check": "Total annotations",
        "value": experiment_annotations_df["annotation_id"].nunique(),
        "expected": 1696,
        "status": (
            "PASS"
            if experiment_annotations_df["annotation_id"].nunique() == 1696
            else "CHECK"
        )
    },
    {
        "check": "Unique annotation IDs",
        "value": experiment_annotations_df["annotation_id"].is_unique,
        "expected": True,
        "status": (
            "PASS"
            if experiment_annotations_df["annotation_id"].is_unique
            else "CHECK"
        )
    },
    {
        "check": "Original pathology labels preserved",
        "value": len(actual_original_labels),
        "expected": len(expected_original_labels),
        "status": (
            "PASS"
            if actual_original_labels == expected_original_labels
            else "CHECK"
        )
    },
    {
        "check": "Binary pathology labels generated",
        "value": len(actual_binary_labels),
        "expected": len(expected_binary_labels),
        "status": (
            "PASS"
            if actual_binary_labels == expected_binary_labels
            else "CHECK"
        )
    },
    {
        "check": "Missing binary labels",
        "value": int(
            experiment_annotations_df["binary_pathology"].isna().sum()
        ),
        "expected": 0,
        "status": (
            "PASS"
            if experiment_annotations_df["binary_pathology"].notna().all()
            else "CHECK"
        )
    },
    {
        "check": "Input strategy candidates",
        "value": len(input_strategy_matrix_df),
        "expected": 5,
        "status": (
            "PASS"
            if len(input_strategy_matrix_df) == 5
            else "CHECK"
        )
    }
])

# 8. Xuất kết quả

output_tables = {
    "nb07_6_original_pathology_summary.csv":
        original_pathology_summary_df,
    "nb07_6_binary_pathology_summary.csv":
        binary_pathology_summary_df,
    "nb07_6_input_strategy_matrix.csv":
        input_strategy_matrix_df,
    "nb07_6_two_stage_pipeline_design.csv":
        pipeline_design_df,
    "nb07_6_experiment_integrity.csv":
        experiment_integrity_df,
    "nb07_6_annotations_with_binary_pathology.csv":
        experiment_annotations_df
}

for filename, dataframe in output_tables.items():
    dataframe.to_csv(
        NB07_OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 9. Hiển thị kết quả

print("PHÂN BỐ PATHOLOGY GỐC")
display(original_pathology_summary_df)

print("\nPHÂN BỐ PATHOLOGY NHỊ PHÂN")
display(binary_pathology_summary_df)

print("\nMA TRẬN CHIẾN LƯỢC ĐẦU VÀO")
display(input_strategy_matrix_df)

print("\nTHIẾT KẾ TWO-STAGE PIPELINE")
display(pipeline_design_df)

print("\nKIỂM TRA TÍNH TOÀN VẸN")
display(experiment_integrity_df)

print(
    "\nTỔNG PASS:",
    (experiment_integrity_df["status"] == "PASS").sum()
)

print(
    "TỔNG CHECK:",
    (experiment_integrity_df["status"] == "CHECK").sum()
)

print("\nĐÃ LƯU KẾT QUẢ TẠI:")
print(NB07_OUTPUT_DIR)


PHÂN BỐ PATHOLOGY GỐC


,dataset_split,pathology,annotation_count,patient_count
0,test,BENIGN,194,101
1,test,BENIGN_WITHOUT_CALLBACK,37,20
2,test,MALIGNANT,147,85
3,train,BENIGN,463,243
4,train,BENIGN_WITHOUT_CALLBACK,83,43
5,train,MALIGNANT,508,278
6,validation,BENIGN,114,61
7,validation,BENIGN_WITHOUT_CALLBACK,21,11
8,validation,MALIGNANT,129,68



PHÂN BỐ PATHOLOGY NHỊ PHÂN


,dataset_split,binary_pathology,annotation_count,patient_count
0,test,BENIGN,231,121
1,test,MALIGNANT,147,85
2,train,BENIGN,546,286
3,train,MALIGNANT,508,278
4,validation,BENIGN,135,72
5,validation,MALIGNANT,129,68



MA TRẬN CHIẾN LƯỢC ĐẦU VÀO


,strategy,input_size,input_type,role,spatial_coverage,relative_pixels_vs_640,priority,evaluation_purpose
0,Full Image 640,640,Full Image,Baseline,Entire mammogram,1.00,Required,Baseline detection performance
1,Full Image 1024,1024,Full Image,High-resolution full-image,Entire mammogram,2.56,High,Evaluate improved lesion detail
2,Full Image 1280,1280,Full Image,High-resolution full-image,Entire mammogram,4.00,Conditional,Evaluate high-resolution feasibility
3,Patch 1024 overlap 20%,1024,Patch,Local-region detection,Local image regions,2.56,High,Evaluate local-detail preservation
4,Patch 1280 overlap 20%,1280,Patch,Local-region detection,Local image regions,4.00,Conditional,Evaluate larger local context



THIẾT KẾ TWO-STAGE PIPELINE


,stage,task,candidate_model,input,target,output,evaluation_metrics
0,Stage 1,Mass Detection and Localization,YOLOv5n,Full Image or image patches,Mass bounding boxes,Bounding boxes and confidence scores,"Precision, Recall, IoU, mAP@50, mAP@50-95"
1,Stage 2,Binary Pathology Classification,VGG16,Mass crop from predicted bounding box,BENIGN / MALIGNANT,Class probabilities,"Accuracy, Precision, Recall, F1, ROC-AUC"
2,Stage 2,Binary Pathology Classification,ResNet50,Mass crop from predicted bounding box,BENIGN / MALIGNANT,Class probabilities,"Accuracy, Precision, Recall, F1, ROC-AUC"
3,Stage 2,Binary Pathology Classification,InceptionV3,Mass crop from predicted bounding box,BENIGN / MALIGNANT,Class probabilities,"Accuracy, Precision, Recall, F1, ROC-AUC"



KIỂM TRA TÍNH TOÀN VẸN


,check,value,expected,status
0,Total annotations,1696,1696,PASS
1,Unique annotation IDs,True,True,PASS
2,Original pathology labels preserved,3,3,PASS
3,Binary pathology labels generated,2,2,PASS
4,Missing binary labels,0,0,PASS
5,Input strategy candidates,5,5,PASS



TỔNG PASS: 6
TỔNG CHECK: 0

ĐÃ LƯU KẾT QUẢ TẠI:
C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design


### TỔNG HỢP GHI CHÚ VÀ MA TRẬN THIẾT KẾ THỰC NGHIỆM

Notebook 07.6 đã hoàn thành việc thiết lập khung cấu trúc dữ liệu và chiến lược thực nghiệm cho toàn bộ hệ thống *Two-stage Pipeline*.

---

### 1. Ánh xạ nhãn Pathology (Binary Classification)
* **Quy tắc gộp nhãn**: Gộp `BENIGN` và `BENIGN_WITHOUT_CALLBACK` thành lớp `BENIGN`; giữ nguyên `MALIGNANT`.
* **Phân phối thực tế (1.696 annotations)**:
  * **BENIGN**: 912 mẫu (Train: 546 | Val: 135 | Test: 231)
  * **MALIGNANT**: 784 mẫu (Train: 508 | Val: 129 | Test: 147)
* **Nguyên tắc kỹ thuật**: 
  * Bảo toàn nguyên vẹn 3 nhãn gốc; lưu nhãn nhị phân vào cột độc lập `binary_pathology` (không ghi đè).
  * Giữ vững nguyên tắc phân chia tập dữ liệu theo **Patient ID** để tránh rò rỉ dữ liệu (*Leakage*).

---

### 2. Chiến lược đầu vào (Input Strategies)
Ma trận 5 phương án thử nghiệm được thiết lập, trong đó tập trung so sánh đối chứng hai hướng tiếp cận chính:
* **Full Image 640**: Baseline bắt buộc (chuẩn hóa kích thước thông thường).
* **Full Image 1024 (Ưu tiên)**: Đánh giá khả năng giữ thông tin chi tiết Mass khi tăng độ phân giải.
* **Patch 1024 (Đối chứng)**: Khảo sát hiệu quả cắt patch (`overlap 20%`) để giữ chi tiết cục bộ (lưu ý rủi ro mất ngữ cảnh giải phẫu và biên tổn thương).
* **Full Image 1280 & Patch 1280**: Phương án dự phòng tùy thuộc vào năng lực phần cứng thực tế (VRAM RTX 4050 6GB).

---

### 3. Kiến trúc Two-Stage Pipeline
* **Stage 1 (Detection & Localization)**: Sử dụng **YOLOv5n** trên ảnh đầu vào (Full Image / Patches) để sinh `Bounding Boxes + Confidence Scores`.
* **Stage 2 (Pathology Classification)**: So sánh hiệu suất của 3 mạng CNN (**VGG16**, **ResNet50**, **InceptionV3**) trên vùng Mass được cắt ra.
* **Lưu ý đánh giá Stage 2**: Sai số định vị từ Stage 1 có thể ảnh hưởng đến kết quả phân loại; do đó cần có đánh giá song song giữa crop từ *Ground Truth* và crop từ *Predicted Bounding Boxes*.

---

### 4. Kết luận trạng thái
* **Kiểm định tính toàn vẹn**: Đạt **6/6 PASS** (không có lỗi hay cảnh báo).
* **Định hướng tiếp theo**: Các ma trận thiết kế này chính thức là cơ sở kỹ thuật vững chắc để bước vào giai đoạn triển khai thực nghiệm và huấn luyện mô hình.

In [14]:
# 07.7 — Tổng hợp thiết kế thực nghiệm

# 1. Cấu hình đầu ra
OUTPUT_DIR = Path(
    r"C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design"
)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 2. Kiến trúc tổng thể
pipeline_design_df = pd.DataFrame([
    {
        "stage": "Stage 1",
        "task": "Mass Detection and Localization",
        "model": "YOLOv5n",
        "input": "Full Image hoặc Patch",
        "target": "Mass Bounding Box",
        "output": "Bounding Box và Detection Confidence",
        "primary_metrics": "Precision, Recall, IoU, mAP@50, mAP@50-95"
    },
    {
        "stage": "Stage 2",
        "task": "Pathology Classification",
        "model": "VGG16",
        "input": "Mass Crop từ Ground Truth hoặc Predicted Bounding Box",
        "target": "BENIGN / MALIGNANT",
        "output": "Class Probability",
        "primary_metrics": "Accuracy, Precision, Recall, F1-score, ROC-AUC"
    },
    {
        "stage": "Stage 2",
        "task": "Pathology Classification",
        "model": "ResNet50",
        "input": "Mass Crop từ Ground Truth hoặc Predicted Bounding Box",
        "target": "BENIGN / MALIGNANT",
        "output": "Class Probability",
        "primary_metrics": "Accuracy, Precision, Recall, F1-score, ROC-AUC"
    },
    {
        "stage": "Stage 2",
        "task": "Pathology Classification",
        "model": "InceptionV3",
        "input": "Mass Crop từ Ground Truth hoặc Predicted Bounding Box",
        "target": "BENIGN / MALIGNANT",
        "output": "Class Probability",
        "primary_metrics": "Accuracy, Precision, Recall, F1-score, ROC-AUC"
    }
])

# 3. Chiến lược đầu vào
input_strategy_design_df = pd.DataFrame([
    {
        "strategy": "Full Image 640",
        "role": "Baseline",
        "input_type": "Toàn bộ ảnh X-quang",
        "resolution": 640,
        "overlap": None,
        "priority": "Bắt buộc"
    },
    {
        "strategy": "Full Image 1024",
        "role": "Resolution comparison",
        "input_type": "Toàn bộ ảnh X-quang",
        "resolution": 1024,
        "overlap": None,
        "priority": "Ưu tiên cao"
    },
    {
        "strategy": "Full Image 1280",
        "role": "Conditional comparison",
        "input_type": "Toàn bộ ảnh X-quang",
        "resolution": 1280,
        "overlap": None,
        "priority": "Tùy tài nguyên"
    },
    {
        "strategy": "Patch 1024",
        "role": "Local-region comparison",
        "input_type": "Các vùng ảnh được chia thành patch",
        "resolution": 1024,
        "overlap": 0.20,
        "priority": "Ưu tiên so sánh"
    },
    {
        "strategy": "Patch 1280",
        "role": "Conditional comparison",
        "input_type": "Các vùng ảnh được chia thành patch",
        "resolution": 1280,
        "overlap": 0.20,
        "priority": "Tùy tài nguyên"
    }
])

# 4. Quy ước nhãn
pathology_design_df = pd.DataFrame([
    {
        "original_label": "BENIGN",
        "binary_label": "BENIGN",
        "binary_label_id": 0,
        "description": "Giữ nguyên nhãn lành tính"
    },
    {
        "original_label": "BENIGN_WITHOUT_CALLBACK",
        "binary_label": "BENIGN",
        "binary_label_id": 0,
        "description": "Gộp vào nhóm BENIGN trong thí nghiệm nhị phân"
    },
    {
        "original_label": "MALIGNANT",
        "binary_label": "MALIGNANT",
        "binary_label_id": 1,
        "description": "Giữ nguyên nhóm ác tính"
    }
])

# 5. Phân chia dữ liệu
split_design_df = pd.DataFrame([
    {
        "split": "Train",
        "annotation_count": 1054,
        "image_count": 982,
        "patient_count": 552
    },
    {
        "split": "Validation",
        "annotation_count": 264,
        "image_count": 249,
        "patient_count": 139
    },
    {
        "split": "Test",
        "annotation_count": 378,
        "image_count": 361,
        "patient_count": 201
    }
])

# 6. Nguyên tắc đánh giá
evaluation_design_df = pd.DataFrame([
    {
        "stage": "Detection",
        "metrics": "Precision, Recall, IoU, mAP@50, mAP@50-95",
        "evaluation_focus": "Khả năng phát hiện và định vị Mass"
    },
    {
        "stage": "Classification",
        "metrics": "Accuracy, Precision, Recall, F1-score, ROC-AUC, Confusion Matrix",
        "evaluation_focus": "Phân loại BENIGN và MALIGNANT"
    },
    {
        "stage": "Two-stage",
        "metrics": "Detection-to-classification performance",
        "evaluation_focus": "Đánh giá ảnh hưởng của sai lệch Bounding Box dự đoán đến phân loại"
    }
])

# 7. Cấu hình phần cứng và nguyên tắc thử nghiệm
experiment_constraints_df = pd.DataFrame([
    {
        "category": "GPU",
        "configuration": "NVIDIA GeForce RTX 4050 Laptop GPU",
        "value": "6 GB VRAM"
    },
    {
        "category": "Framework",
        "configuration": "PyTorch",
        "value": "2.5.1+cu121"
    },
    {
        "category": "Python",
        "configuration": "Python",
        "value": "3.11.18"
    },
    {
        "category": "Batch size",
        "configuration": "Khởi đầu",
        "value": "1 hoặc 2, điều chỉnh theo VRAM thực tế"
    },
    {
        "category": "Precision",
        "configuration": "Mixed precision",
        "value": "Thử nghiệm FP16 nếu tương thích"
    },
    {
        "category": "Model selection",
        "configuration": "Validation set",
        "value": "Dùng để lựa chọn cấu hình; không dùng Test để tinh chỉnh"
    },
    {
        "category": "Reproducibility",
        "configuration": "Random seed và cấu hình huấn luyện",
        "value": "Ghi nhận nhất quán giữa các lần thử nghiệm"
    }
])

# 8. Kiểm tra tính nhất quán của thiết kế
checks = []

def add_check(name, condition, detail):
    checks.append({
        "check": name,
        "status": "PASS" if bool(condition) else "CHECK",
        "detail": detail
    })

add_check(
    "Two-stage pipeline",
    pipeline_design_df["stage"].nunique() == 2,
    "Bao gồm Detection và Classification"
)

add_check(
    "YOLOv5n detection",
    ((pipeline_design_df["stage"] == "Stage 1") &
     (pipeline_design_df["model"] == "YOLOv5n")).any(),
    "YOLOv5n đảm nhiệm phát hiện và định vị Mass"
)

add_check(
    "Three classification candidates",
    set(pipeline_design_df.loc[
        pipeline_design_df["stage"] == "Stage 2", "model"
    ]) == {"VGG16", "ResNet50", "InceptionV3"},
    "Có đủ ba kiến trúc CNN để so sánh"
)

add_check(
    "Original pathology labels preserved",
    set(pathology_design_df["original_label"]) ==
    {"BENIGN", "BENIGN_WITHOUT_CALLBACK", "MALIGNANT"},
    "Giữ nguyên ba nhãn gốc trong dữ liệu"
)

add_check(
    "Binary pathology mapping",
    set(pathology_design_df["binary_label"]) == {"BENIGN", "MALIGNANT"},
    "Ánh xạ nhị phân có hai nhóm"
)

add_check(
    "Input strategies",
    len(input_strategy_design_df) == 5,
    "Có năm chiến lược đầu vào được xem xét"
)

add_check(
    "Patient-level split",
    split_design_df["patient_count"].sum() == 892,
    "Tổng số Patient ID theo ba tập là 892"
)

add_check(
    "Annotation total",
    split_design_df["annotation_count"].sum() == 1696,
    "Tổng số annotation theo ba tập là 1.696"
)

add_check(
    "Image total",
    split_design_df["image_count"].sum() == 1592,
    "Tổng số ảnh duy nhất theo ba tập là 1.592"
)

add_check(
    "Hardware configuration",
    True,
    "RTX 4050 Laptop GPU 6 GB VRAM; cấu hình huấn luyện cần kiểm chứng thực tế"
)

design_integrity_df = pd.DataFrame(checks)

# 9. Xuất các bảng thiết kế
export_tables = {
    "nb07_7_pipeline_design.csv": pipeline_design_df,
    "nb07_7_input_strategy_design.csv": input_strategy_design_df,
    "nb07_7_pathology_label_design.csv": pathology_design_df,
    "nb07_7_split_design.csv": split_design_df,
    "nb07_7_evaluation_design.csv": evaluation_design_df,
    "nb07_7_experiment_constraints.csv": experiment_constraints_df,
    "nb07_7_design_integrity_report.csv": design_integrity_df
}

for filename, dataframe in export_tables.items():
    dataframe.to_csv(
        OUTPUT_DIR / filename,
        index=False,
        encoding="utf-8-sig"
    )

# 10. Tổng hợp kết quả
print("=== NOTEBOOK 07.7 — EXPERIMENT DESIGN SUMMARY ===")
print(f"Output directory: {OUTPUT_DIR}")
print(f"Pipeline stages: {pipeline_design_df['stage'].nunique()}")
print(f"Classification models: {pipeline_design_df.loc[pipeline_design_df['stage'] == 'Stage 2', 'model'].nunique()}")
print(f"Input strategies: {len(input_strategy_design_df)}")
print(f"Original pathology classes: {pathology_design_df['original_label'].nunique()}")
print(f"Binary pathology classes: {pathology_design_df['binary_label'].nunique()}")
print(f"Total annotations: {split_design_df['annotation_count'].sum()}")
print(f"Total unique images: {split_design_df['image_count'].sum()}")
print(f"Total Patient IDs across splits: {split_design_df['patient_count'].sum()}")

print("\n=== DESIGN INTEGRITY ===")
display(design_integrity_df)

print("\n=== PIPELINE DESIGN ===")
display(pipeline_design_df)

print("\n=== INPUT STRATEGY ===")
display(input_strategy_design_df)

print("\n=== PATHOLOGY LABEL MAPPING ===")
display(pathology_design_df)

print("\n=== DATA SPLIT ===")
display(split_design_df)

print("\n=== EVALUATION PLAN ===")
display(evaluation_design_df)

print("\n=== EXPERIMENT CONSTRAINTS ===")
display(experiment_constraints_df)

print("\n=== EXPORT ===")
for filename in export_tables:
    print(f"Saved: {OUTPUT_DIR / filename}")

print(
    f"\nIntegrity result: "
    f"{(design_integrity_df['status'] == 'PASS').sum()}/"
    f"{len(design_integrity_df)} PASS"
)


=== NOTEBOOK 07.7 — EXPERIMENT DESIGN SUMMARY ===
Output directory: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design
Pipeline stages: 2
Classification models: 3
Input strategies: 5
Original pathology classes: 3
Binary pathology classes: 2
Total annotations: 1696
Total unique images: 1592
Total Patient IDs across splits: 892

=== DESIGN INTEGRITY ===


,check,status,detail
0,Two-stage pipeline,PASS,Bao gồm Detection và Classification
1,YOLOv5n detection,PASS,YOLOv5n đảm nhiệm phát hiện và định vị Mass
2,Three classification candidates,PASS,Có đủ ba kiến trúc CNN để so sánh
3,Original pathology labels preserved,PASS,Giữ nguyên ba nhãn gốc trong dữ liệu
4,Binary pathology mapping,PASS,Ánh xạ nhị phân có hai nhóm
5,Input strategies,PASS,Có năm chiến lược đầu vào được xem xét
6,Patient-level split,PASS,Tổng số Patient ID theo ba tập là 892
7,Annotation total,PASS,Tổng số annotation theo ba tập là 1.696
8,Image total,PASS,Tổng số ảnh duy nhất theo ba tập là 1.592
9,Hardware configuration,PASS,RTX 4050 Laptop GPU 6 GB VRAM; cấu hình huấn l...



=== PIPELINE DESIGN ===


,stage,task,model,input,target,output,primary_metrics
0,Stage 1,Mass Detection and Localization,YOLOv5n,Full Image hoặc Patch,Mass Bounding Box,Bounding Box và Detection Confidence,"Precision, Recall, IoU, mAP@50, mAP@50-95"
1,Stage 2,Pathology Classification,VGG16,Mass Crop từ Ground Truth hoặc Predicted Bound...,BENIGN / MALIGNANT,Class Probability,"Accuracy, Precision, Recall, F1-score, ROC-AUC"
2,Stage 2,Pathology Classification,ResNet50,Mass Crop từ Ground Truth hoặc Predicted Bound...,BENIGN / MALIGNANT,Class Probability,"Accuracy, Precision, Recall, F1-score, ROC-AUC"
3,Stage 2,Pathology Classification,InceptionV3,Mass Crop từ Ground Truth hoặc Predicted Bound...,BENIGN / MALIGNANT,Class Probability,"Accuracy, Precision, Recall, F1-score, ROC-AUC"



=== INPUT STRATEGY ===


,strategy,role,input_type,resolution,overlap,priority
0,Full Image 640,Baseline,Toàn bộ ảnh X-quang,640,NaN,Bắt buộc
1,Full Image 1024,Resolution comparison,Toàn bộ ảnh X-quang,1024,NaN,Ưu tiên cao
2,Full Image 1280,Conditional comparison,Toàn bộ ảnh X-quang,1280,NaN,Tùy tài nguyên
3,Patch 1024,Local-region comparison,Các vùng ảnh được chia thành patch,1024,0.2,Ưu tiên so sánh
4,Patch 1280,Conditional comparison,Các vùng ảnh được chia thành patch,1280,0.2,Tùy tài nguyên


... (5 outputs left out)

,stage,metrics,evaluation_focus
0,Detection,"Precision, Recall, IoU, mAP@50, mAP@50-95",Khả năng phát hiện và định vị Mass
1,Classification,"Accuracy, Precision, Recall, F1-score, ROC-AUC...",Phân loại BENIGN và MALIGNANT
2,Two-stage,Detection-to-classification performance,Đánh giá ảnh hưởng của sai lệch Bounding Box d...



=== EXPERIMENT CONSTRAINTS ===


,category,configuration,value
0,GPU,NVIDIA GeForce RTX 4050 Laptop GPU,6 GB VRAM
1,Framework,PyTorch,2.5.1+cu121
2,Python,Python,3.11.18
3,Batch size,Khởi đầu,"1 hoặc 2, điều chỉnh theo VRAM thực tế"
4,Precision,Mixed precision,Thử nghiệm FP16 nếu tương thích
5,Model selection,Validation set,Dùng để lựa chọn cấu hình; không dùng Test để ...
6,Reproducibility,Random seed và cấu hình huấn luyện,Ghi nhận nhất quán giữa các lần thử nghiệm



=== EXPORT ===
Saved: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design\nb07_7_pipeline_design.csv
Saved: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design\nb07_7_input_strategy_design.csv
Saved: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design\nb07_7_pathology_label_design.csv
Saved: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design\nb07_7_split_design.csv
Saved: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design\nb07_7_evaluation_design.csv
Saved: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design\nb07_7_experiment_constraints.csv
Saved: C:\Breast-Cancer-CBIS-DDSM\Data\processed\nb07_model_design\nb07_7_design_integrity_report.csv

Integrity result: 10/10 PASS


## KẾT LUẬN VÀ ĐỊNH HƯỚNG TRIỂN KHAI

### 1. Tổng quan & Kiến trúc Two-Stage Pipeline
* **Giai đoạn 1 (Detection & Localization)**: Sử dụng **YOLOv5n** để phát hiện và định vị vùng tổn thương (*Mass*). 
  * *Đầu ra*: Bounding Box và Confidence Scores.
  * *Chỉ số*: Precision, Recall, IoU, mAP@50, mAP@50-95.
* **Giai đoạn 2 (Pathology Classification)**: So sánh 3 kiến trúc **VGG16**, **ResNet50** và **InceptionV3** để phân loại nhị phân (`BENIGN` / `MALIGNANT`).
  * *Đầu ra*: Class probabilities.
  * *Chỉ số*: Accuracy, Precision, Recall, F1-score, ROC-AUC, Confusion Matrix.
  * *Chiến lược đánh giá*: Thực hiện trên cả Ground Truth Crop và Predicted Crop để đo lường ảnh hưởng của sai lệch định vị ở Giai đoạn 1.

---

### 2. Chiến lược dữ liệu & Thực nghiệm
* **Quy mô dữ liệu (CBIS-DDSM Mass)**: Tổng số **1.696 annotations** (1.592 ảnh Full Image, 892 Patient IDs), phân chia theo chuẩn Patient-level: **Train (1.054)**, **Validation (264)**, **Test (378)**.
* **Chiến lược đầu vào**: Khảo sát 5 phương án gồm Full Image (640, 1024, 1280) và Patch (1024, 1280 với overlap 20%).
* **Quản lý nhãn**: Bảo toàn 3 nhãn gốc; áp dụng gộp nhãn (`BENIGN` / `MALIGNANT`) vào cột độc lập cho bài toán phân loại nhị phân.

---

### 3. Giới hạn & Nguyên tắc triển khai
* Thiết kế hiện tại là khung định hướng thực nghiệm, cần kiểm chứng trực tiếp trên phần cứng **NVIDIA RTX 4050 Laptop GPU (6 GB VRAM)**.
* **Nguyên tắc cốt lõi**: 
  * Dùng Validation Set để tuning; giữ Test Set hoàn toàn độc lập.
  * Ghi nhận đầy đủ seed, cấu hình, thời gian và mức tiêu thụ VRAM.
  * Báo cáo tách biệt kết quả Detection, Classification và toàn bộ Pipeline.
  * Kết quả mang tính kỹ thuật, không thay thế chẩn đoán y khoa độc lập.

---

### 4. Định hướng Notebook 08: Training Environment & Initial Feasibility
1. Thiết lập và chuẩn hóa môi trường huấn luyện.
2. Chuẩn bị cấu trúc dữ liệu đầu vào chuẩn định dạng YOLO.
3. Thử nghiệm cấu hình YOLOv5n ban đầu với độ phân giải và batch size phù hợp.
4. Kiểm tra khả năng nạp dữ liệu, huấn luyện thử (*smoke test*), sử dụng GPU và lưu checkpoint.
5. Chốt cấu hình khả thi trước khi bước vào các thí nghiệm chính thức.

---

### Trạng thái tổng kết
* **Kết quả**: Vượt qua **10/10 kiểm tra tính nhất quán**.
* **Trạng thái**: <span style="color: green; font-weight: bold;">COMPLETED</span>.
* **Tiếp theo**: Chuyển sang **Notebook 08 — Training Environment and Initial Model Feasibility**.